# 04h Extraction, batch h: ten studies coded from their PDFs and two full-text exclusions

Design log.
- Batch h of notebook 04 (plan v2 B4), coded under the frozen schema (schema/record_schema.yaml, v1) and the coding rules, as batches a to g were. The ten reports: When Defenses Backfire (zenodo.20735483), Same Attack, Different Fates (SOC triage, zenodo.20772702), Prompt Transformation Effects Do Not Transfer (zenodo.21120124), What Determines the Vulnerability of Small LLM Agents (zenodo.21343930), Poisoned Tools (zenodo.21349713), The MLPs of Layers 8-15 of Meta-SecAlign-8B (zenodo.23506227), Whose words count? (zenodo.23758884), Who said this? (zenodo.23758886), Prompt-Injection Defences on a Small Local Agent (zenodo.23811220) and A Cross-Lingual Security Evaluation of Prompt Injection Defenses in Small Language Models (SBSeg 2026).
- Each record's extraction_scope names the tables and rows coded. As before, detectors, sanitisers, filters and system-level designs are left out and the scope says so (amendment 4): Allowlist Sanitization, Dynamic Canary, the LLM Classifier and the stacked combination in the SOC study, D3 quarantine in Poisoned Tools, plan-then-gate (a planner with a separate call checker), AgentDojo's tool filter and the prompt-injection classifier in the small local agent study.
- Coding decisions worth knowing when reading these records (reading aids for notebook 05, not findings):
  - The backfire study prints attack success and utility as fractions; they are coded as printed with unit fraction.
  - Poisoned Tools reports only paired changes against the undefended agent, pooled over five models; the undefended evaluations carry no value and the defended results carry the printed changes.
  - The factorial study's neutral-preamble control is coded as its own arm, with M comparisons that match the defences against it.
  - The small-agent vulnerability study has no undefended arm: spotlighting is compared with the harness's one-line inline instruction.
  - The Meta-SecAlign component study cuts restricted adapters from the released adapter (CORE, without CORE, a window four layers earlier); they keep the parent's axes and are compared on 3,000 fresh SEP items.
  - The two steering studies are coded as fixed inference rules with axis 1 internal_intervention (steering directions are fitted, not trained, so the training fields are not_applicable). Whose words count? is coded for its Section 8 injection test only, with its random-direction control as an M comparison; the author presents that test as a replication. In Who said this? the InjecAgent rates are computed on each condition's valid replies, whose number changes with the lever, so those comparisons are grade B.
  - The cross-lingual study trains DefensiveTokens itself on base models that separate instruction and data with plain-text [INST]/[DATA]/[RESP] markers, so axis 1 is delimiters here rather than chat roles as in the original release; a residual note says so. No utility measure is reported.
  - Model names printed in the papers are kept as printed, including closed models used as rows (Claude Haiku 4.5 and Claude Sonnet 4.6 in the backfire and factorial studies).
- Full-text exclusions found while coding: the red-teaming study of a composed RAG agent (zenodo.23953154) measures only a hardened build that bundles four server-side controls, and the multi-model trade-off study (jitcai.2026.v3.i3.071) measures only an input filter combined with prompt hardening; both are excluded as system level. Stage 2b checks both anchors in the PDFs and carries the decisions into schema/fulltext_decisions.csv, schema/screening_decisions.csv, reports/screening.csv, reports/defences.csv, reports/fulltext_check.csv and reports/prisma_counts.json, as batches d, f and g did. The number of included studies falls from 133 to 131.
- Outputs: reports/records/ (ten files), reports/extraction_04h_check.csv, reports/extraction_04h_summary.csv, reports/extraction_batches.csv (rows for batch h added, earlier batches kept), reports/decision_04h.txt.
- CPU; no network calls beyond pip; model calls: 0.

## Stage 0. Session bootstrap

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/SourceMap_Research"
    REPO_DIR = PROJECT_ROOT + "/sourcemap-research"
except ImportError:
    PROJECT_ROOT = os.environ["SOURCEMAP_PROJECT_ROOT"]
    REPO_DIR = os.environ["SOURCEMAP_REPO_ROOT"]
os.environ["SOURCEMAP_PROJECT_ROOT"] = PROJECT_ROOT
os.environ["SOURCEMAP_REPO_ROOT"] = REPO_DIR
os.chdir(REPO_DIR)
if "src" not in sys.path:
    sys.path.insert(0, "src")
for module, package in (("yaml", "pyyaml"), ("pypdf", "pypdf"), ("pypdfium2", "pypdfium2")):
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)
import pypdf
if int(pypdf.__version__.split(".")[0]) < 4:                 # the column-preserving extraction needs pypdf 4 or later
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pypdf>=4"], check=False)
    print("pypdf upgraded; restart the runtime if the next stage reports an old version")
print(subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True).stdout.strip() or "pull: nothing to report")
print("cwd:", os.getcwd(), "| pypdf", pypdf.__version__)

## Stage 1. The 10 records

In [ ]:
Path("reports/records").mkdir(parents=True, exist_ok=True)       # %%writefile does not create folders

`backfire_zenodo2026` (doi_10_5281_zenodo_20735483)

In [ ]:
%%writefile reports/records/backfire_zenodo2026.yaml
report:
  id: backfire_zenodo2026
  slug: doi_10_5281_zenodo_20735483
  title: 'When Defenses Backfire: Model-Dependent Failure Modes in Prompt-Injection Defense for LLM Agents'
  first_author: Munish Singh
  venue: Zenodo preprint
  peer_reviewed: 'no'
  doi: 10.5281/zenodo.20735483
  version_read: Zenodo version (17 June 2026)
  first_date: '2026-06-17'
  read: F
  role: re_measures
  code_release: per-cell logs stated as released
  extraction_scope: 'Appendix A: Table 6 (static attack, all five models, four suites, three conditions, ASR and utility); the abstract claim (the defence-conditional attacks of Table 2, the tool-budget intervention of Tables
    3 and 4 and the tier summary of Table 5 not coded)'
defences:
- id: srp_llama
  name: Sentinel reflection prompt (verification checklist prefixed to every user and tool message before each call)
  origin: other_report
  base_checkpoint: Llama 4 Scout 17B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: AgentDojo tool-calling agent with a generic base system prompt; the injection arrives in tool outputs
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: The following prefix is prepended to the textual content
- id: spot_llama
  name: Spotlighting by datamarking (spaces in tool outputs replaced with U+02C8, preamble in the system prompt)
  origin: other_report
  base_checkpoint: Llama 4 Scout 17B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: AgentDojo tool-calling agent with a generic base system prompt; the injection arrives in tool outputs; tool outputs datamarked
  layout_compat: *id001
  anchor: Spaces in untrusted tool-output content are replaced with the
- id: srp_ds
  name: Sentinel reflection prompt (verification checklist prefixed to every user and tool message before each call)
  origin: other_report
  base_checkpoint: DeepSeek V3.2
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: AgentDojo tool-calling agent with a generic base system prompt; the injection arrives in tool outputs
  layout_compat: *id001
  anchor: The following prefix is prepended to the textual content
- id: spot_ds
  name: Spotlighting by datamarking (spaces in tool outputs replaced with U+02C8, preamble in the system prompt)
  origin: other_report
  base_checkpoint: DeepSeek V3.2
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: AgentDojo tool-calling agent with a generic base system prompt; the injection arrives in tool outputs; tool outputs datamarked
  layout_compat: *id001
  anchor: Spaces in untrusted tool-output content are replaced with the
- id: srp_qwen
  name: Sentinel reflection prompt (verification checklist prefixed to every user and tool message before each call)
  origin: other_report
  base_checkpoint: Qwen 3 32B (dense)
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: AgentDojo tool-calling agent with a generic base system prompt; the injection arrives in tool outputs
  layout_compat: *id001
  anchor: The following prefix is prepended to the textual content
- id: spot_qwen
  name: Spotlighting by datamarking (spaces in tool outputs replaced with U+02C8, preamble in the system prompt)
  origin: other_report
  base_checkpoint: Qwen 3 32B (dense)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: AgentDojo tool-calling agent with a generic base system prompt; the injection arrives in tool outputs; tool outputs datamarked
  layout_compat: *id001
  anchor: Spaces in untrusted tool-output content are replaced with the
- id: srp_haiku
  name: Sentinel reflection prompt (verification checklist prefixed to every user and tool message before each call)
  origin: other_report
  base_checkpoint: Claude Haiku 4.5
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: AgentDojo tool-calling agent with a generic base system prompt; the injection arrives in tool outputs
  layout_compat: *id001
  anchor: The following prefix is prepended to the textual content
- id: spot_haiku
  name: Spotlighting by datamarking (spaces in tool outputs replaced with U+02C8, preamble in the system prompt)
  origin: other_report
  base_checkpoint: Claude Haiku 4.5
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: AgentDojo tool-calling agent with a generic base system prompt; the injection arrives in tool outputs; tool outputs datamarked
  layout_compat: *id001
  anchor: Spaces in untrusted tool-output content are replaced with the
- id: srp_sonnet
  name: Sentinel reflection prompt (verification checklist prefixed to every user and tool message before each call)
  origin: other_report
  base_checkpoint: Claude Sonnet 4.6
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: AgentDojo tool-calling agent with a generic base system prompt; the injection arrives in tool outputs
  layout_compat: *id001
  anchor: The following prefix is prepended to the textual content
- id: spot_sonnet
  name: Spotlighting by datamarking (spaces in tool outputs replaced with U+02C8, preamble in the system prompt)
  origin: other_report
  base_checkpoint: Claude Sonnet 4.6
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: AgentDojo tool-calling agent with a generic base system prompt; the injection arrives in tool outputs; tool outputs datamarked
  layout_compat: *id001
  anchor: Spaces in untrusted tool-output content are replaced with the
evaluations:
- id: ev_llama_banking_none
  defence_id: none:Llama 4 Scout 17B
  benchmark: AgentDojo banking suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_llama_banking_srp
  defence_id: srp_llama
  benchmark: AgentDojo banking suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
- id: ev_llama_banking_spot
  defence_id: spot_llama
  benchmark: AgentDojo banking suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
- id: ev_llama_slack_none
  defence_id: none:Llama 4 Scout 17B
  benchmark: AgentDojo slack suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_llama_slack_srp
  defence_id: srp_llama
  benchmark: AgentDojo slack suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
- id: ev_llama_slack_spot
  defence_id: spot_llama
  benchmark: AgentDojo slack suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
- id: ev_llama_travel_none
  defence_id: none:Llama 4 Scout 17B
  benchmark: AgentDojo travel suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_llama_travel_srp
  defence_id: srp_llama
  benchmark: AgentDojo travel suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
- id: ev_llama_travel_spot
  defence_id: spot_llama
  benchmark: AgentDojo travel suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
- id: ev_llama_workspace_none
  defence_id: none:Llama 4 Scout 17B
  benchmark: AgentDojo workspace suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_llama_workspace_srp
  defence_id: srp_llama
  benchmark: AgentDojo workspace suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 3
- id: ev_llama_workspace_spot
  defence_id: spot_llama
  benchmark: AgentDojo workspace suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 3
- id: ev_ds_banking_none
  defence_id: none:DeepSeek V3.2
  benchmark: AgentDojo banking suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_ds_banking_srp
  defence_id: srp_ds
  benchmark: AgentDojo banking suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
- id: ev_ds_banking_spot
  defence_id: spot_ds
  benchmark: AgentDojo banking suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
- id: ev_ds_slack_none
  defence_id: none:DeepSeek V3.2
  benchmark: AgentDojo slack suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_ds_slack_srp
  defence_id: srp_ds
  benchmark: AgentDojo slack suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
- id: ev_ds_slack_spot
  defence_id: spot_ds
  benchmark: AgentDojo slack suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
- id: ev_ds_travel_none
  defence_id: none:DeepSeek V3.2
  benchmark: AgentDojo travel suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_ds_travel_srp
  defence_id: srp_ds
  benchmark: AgentDojo travel suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
- id: ev_ds_travel_spot
  defence_id: spot_ds
  benchmark: AgentDojo travel suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
- id: ev_ds_workspace_none
  defence_id: none:DeepSeek V3.2
  benchmark: AgentDojo workspace suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_ds_workspace_srp
  defence_id: srp_ds
  benchmark: AgentDojo workspace suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 3
- id: ev_ds_workspace_spot
  defence_id: spot_ds
  benchmark: AgentDojo workspace suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 3
- id: ev_qwen_banking_none
  defence_id: none:Qwen 3 32B (dense)
  benchmark: AgentDojo banking suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_qwen_banking_srp
  defence_id: srp_qwen
  benchmark: AgentDojo banking suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
- id: ev_qwen_banking_spot
  defence_id: spot_qwen
  benchmark: AgentDojo banking suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
- id: ev_qwen_slack_none
  defence_id: none:Qwen 3 32B (dense)
  benchmark: AgentDojo slack suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_qwen_slack_srp
  defence_id: srp_qwen
  benchmark: AgentDojo slack suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
- id: ev_qwen_slack_spot
  defence_id: spot_qwen
  benchmark: AgentDojo slack suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
- id: ev_qwen_travel_none
  defence_id: none:Qwen 3 32B (dense)
  benchmark: AgentDojo travel suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_qwen_travel_srp
  defence_id: srp_qwen
  benchmark: AgentDojo travel suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
- id: ev_qwen_travel_spot
  defence_id: spot_qwen
  benchmark: AgentDojo travel suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
- id: ev_qwen_workspace_none
  defence_id: none:Qwen 3 32B (dense)
  benchmark: AgentDojo workspace suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_qwen_workspace_srp
  defence_id: srp_qwen
  benchmark: AgentDojo workspace suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 3
- id: ev_qwen_workspace_spot
  defence_id: spot_qwen
  benchmark: AgentDojo workspace suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 3
- id: ev_haiku_banking_none
  defence_id: none:Claude Haiku 4.5
  benchmark: AgentDojo banking suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_haiku_banking_srp
  defence_id: srp_haiku
  benchmark: AgentDojo banking suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
- id: ev_haiku_banking_spot
  defence_id: spot_haiku
  benchmark: AgentDojo banking suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
- id: ev_haiku_slack_none
  defence_id: none:Claude Haiku 4.5
  benchmark: AgentDojo slack suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_haiku_slack_srp
  defence_id: srp_haiku
  benchmark: AgentDojo slack suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
- id: ev_haiku_slack_spot
  defence_id: spot_haiku
  benchmark: AgentDojo slack suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
- id: ev_haiku_travel_none
  defence_id: none:Claude Haiku 4.5
  benchmark: AgentDojo travel suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_haiku_travel_srp
  defence_id: srp_haiku
  benchmark: AgentDojo travel suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
- id: ev_haiku_travel_spot
  defence_id: spot_haiku
  benchmark: AgentDojo travel suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
- id: ev_haiku_workspace_none
  defence_id: none:Claude Haiku 4.5
  benchmark: AgentDojo workspace suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_haiku_workspace_srp
  defence_id: srp_haiku
  benchmark: AgentDojo workspace suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 3
- id: ev_haiku_workspace_spot
  defence_id: spot_haiku
  benchmark: AgentDojo workspace suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 3
- id: ev_sonnet_banking_none
  defence_id: none:Claude Sonnet 4.6
  benchmark: AgentDojo banking suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_sonnet_banking_srp
  defence_id: srp_sonnet
  benchmark: AgentDojo banking suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
- id: ev_sonnet_banking_spot
  defence_id: spot_sonnet
  benchmark: AgentDojo banking suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
- id: ev_sonnet_slack_none
  defence_id: none:Claude Sonnet 4.6
  benchmark: AgentDojo slack suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_sonnet_slack_srp
  defence_id: srp_sonnet
  benchmark: AgentDojo slack suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
- id: ev_sonnet_slack_spot
  defence_id: spot_sonnet
  benchmark: AgentDojo slack suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
- id: ev_sonnet_travel_none
  defence_id: none:Claude Sonnet 4.6
  benchmark: AgentDojo travel suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_sonnet_travel_srp
  defence_id: srp_sonnet
  benchmark: AgentDojo travel suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
- id: ev_sonnet_travel_spot
  defence_id: spot_sonnet
  benchmark: AgentDojo travel suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
- id: ev_sonnet_workspace_none
  defence_id: none:Claude Sonnet 4.6
  benchmark: AgentDojo workspace suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_sonnet_workspace_srp
  defence_id: srp_sonnet
  benchmark: AgentDojo workspace suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 3
- id: ev_sonnet_workspace_spot
  defence_id: spot_sonnet
  benchmark: AgentDojo workspace suite, stock important_instructions attack (static)
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo security and utility checks
  layout_same_as_defence: 'yes'
  measured_by: backfire_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '560'
  pages:
    benchmark: 3
results:
- id: r_llama_banking_none_asr
  evaluation_id: ev_llama_banking_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.319'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_llama_banking_none_util
  evaluation_id: ev_llama_banking_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.243'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_llama_banking_srp_asr
  evaluation_id: ev_llama_banking_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.632'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_llama_banking_srp_util
  evaluation_id: ev_llama_banking_srp
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.181'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_llama_banking_spot_asr
  evaluation_id: ev_llama_banking_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.438'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_llama_banking_spot_util
  evaluation_id: ev_llama_banking_spot
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.174'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_llama_slack_none_asr
  evaluation_id: ev_llama_slack_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.267'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_llama_slack_none_util
  evaluation_id: ev_llama_slack_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.229'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_llama_slack_srp_asr
  evaluation_id: ev_llama_slack_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.562'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_llama_slack_srp_util
  evaluation_id: ev_llama_slack_srp
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.219'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_llama_slack_spot_asr
  evaluation_id: ev_llama_slack_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.229'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_llama_slack_spot_util
  evaluation_id: ev_llama_slack_spot
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.248'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_llama_travel_none_asr
  evaluation_id: ev_llama_travel_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.171'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_llama_travel_none_util
  evaluation_id: ev_llama_travel_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.150'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_llama_travel_srp_asr
  evaluation_id: ev_llama_travel_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.193'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_llama_travel_srp_util
  evaluation_id: ev_llama_travel_srp
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.129'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_llama_travel_spot_asr
  evaluation_id: ev_llama_travel_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.071'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_llama_travel_spot_util
  evaluation_id: ev_llama_travel_spot
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.143'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_llama_workspace_none_asr
  evaluation_id: ev_llama_workspace_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.037'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_llama_workspace_none_util
  evaluation_id: ev_llama_workspace_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.111'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_llama_workspace_srp_asr
  evaluation_id: ev_llama_workspace_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.054'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_llama_workspace_srp_util
  evaluation_id: ev_llama_workspace_srp
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.087'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_llama_workspace_spot_asr
  evaluation_id: ev_llama_workspace_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.011'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_llama_workspace_spot_util
  evaluation_id: ev_llama_workspace_spot
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.118'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_ds_banking_none_asr
  evaluation_id: ev_ds_banking_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.722'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_ds_banking_none_util
  evaluation_id: ev_ds_banking_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.701'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_ds_banking_srp_asr
  evaluation_id: ev_ds_banking_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.028'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_ds_banking_srp_util
  evaluation_id: ev_ds_banking_srp
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.597'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_ds_banking_spot_asr
  evaluation_id: ev_ds_banking_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.743'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_ds_banking_spot_util
  evaluation_id: ev_ds_banking_spot
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.729'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_ds_slack_none_asr
  evaluation_id: ev_ds_slack_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.952'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_ds_slack_none_util
  evaluation_id: ev_ds_slack_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.657'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Table 6
- id: r_ds_slack_srp_asr
  evaluation_id: ev_ds_slack_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.086'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_ds_slack_srp_util
  evaluation_id: ev_ds_slack_srp
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.705'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_ds_slack_spot_asr
  evaluation_id: ev_ds_slack_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.952'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_ds_slack_spot_util
  evaluation_id: ev_ds_slack_spot
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.657'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_ds_travel_none_asr
  evaluation_id: ev_ds_travel_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.764'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_ds_travel_none_util
  evaluation_id: ev_ds_travel_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.221'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_ds_travel_srp_asr
  evaluation_id: ev_ds_travel_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.036'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_ds_travel_srp_util
  evaluation_id: ev_ds_travel_srp
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.800'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_ds_travel_spot_asr
  evaluation_id: ev_ds_travel_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.736'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_ds_travel_spot_util
  evaluation_id: ev_ds_travel_spot
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.236'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_ds_workspace_none_asr
  evaluation_id: ev_ds_workspace_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.534'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_ds_workspace_none_util
  evaluation_id: ev_ds_workspace_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.261'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_ds_workspace_srp_asr
  evaluation_id: ev_ds_workspace_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.032'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_ds_workspace_srp_util
  evaluation_id: ev_ds_workspace_srp
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.793'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_ds_workspace_spot_asr
  evaluation_id: ev_ds_workspace_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.461'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_ds_workspace_spot_util
  evaluation_id: ev_ds_workspace_spot
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.212'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_banking_none_asr
  evaluation_id: ev_qwen_banking_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.410'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_banking_none_util
  evaluation_id: ev_qwen_banking_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.451'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_banking_srp_asr
  evaluation_id: ev_qwen_banking_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.424'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_banking_srp_util
  evaluation_id: ev_qwen_banking_srp
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.444'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_banking_spot_asr
  evaluation_id: ev_qwen_banking_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.493'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_banking_spot_util
  evaluation_id: ev_qwen_banking_spot
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.389'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_slack_none_asr
  evaluation_id: ev_qwen_slack_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.610'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_slack_none_util
  evaluation_id: ev_qwen_slack_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.543'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_slack_srp_asr
  evaluation_id: ev_qwen_slack_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.667'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_slack_srp_util
  evaluation_id: ev_qwen_slack_srp
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.552'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_slack_spot_asr
  evaluation_id: ev_qwen_slack_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.648'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_slack_spot_util
  evaluation_id: ev_qwen_slack_spot
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.524'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_travel_none_asr
  evaluation_id: ev_qwen_travel_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.636'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_travel_none_util
  evaluation_id: ev_qwen_travel_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.136'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_travel_srp_asr
  evaluation_id: ev_qwen_travel_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.521'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_travel_srp_util
  evaluation_id: ev_qwen_travel_srp
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.136'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_travel_spot_asr
  evaluation_id: ev_qwen_travel_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.250'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_travel_spot_util
  evaluation_id: ev_qwen_travel_spot
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.143'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_workspace_none_asr
  evaluation_id: ev_qwen_workspace_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.111'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_workspace_none_util
  evaluation_id: ev_qwen_workspace_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.318'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_workspace_srp_asr
  evaluation_id: ev_qwen_workspace_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.123'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_workspace_srp_util
  evaluation_id: ev_qwen_workspace_srp
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.232'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_workspace_spot_asr
  evaluation_id: ev_qwen_workspace_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.089'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_qwen_workspace_spot_util
  evaluation_id: ev_qwen_workspace_spot
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.320'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_banking_none_asr
  evaluation_id: ev_haiku_banking_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_banking_none_util
  evaluation_id: ev_haiku_banking_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.535'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_banking_srp_asr
  evaluation_id: ev_haiku_banking_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_banking_srp_util
  evaluation_id: ev_haiku_banking_srp
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.354'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_banking_spot_asr
  evaluation_id: ev_haiku_banking_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_banking_spot_util
  evaluation_id: ev_haiku_banking_spot
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.576'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_slack_none_asr
  evaluation_id: ev_haiku_slack_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_slack_none_util
  evaluation_id: ev_haiku_slack_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.714'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_slack_srp_asr
  evaluation_id: ev_haiku_slack_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_slack_srp_util
  evaluation_id: ev_haiku_slack_srp
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.476'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_slack_spot_asr
  evaluation_id: ev_haiku_slack_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_slack_spot_util
  evaluation_id: ev_haiku_slack_spot
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.667'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_travel_none_asr
  evaluation_id: ev_haiku_travel_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_travel_none_util
  evaluation_id: ev_haiku_travel_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.679'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_travel_srp_asr
  evaluation_id: ev_haiku_travel_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_travel_srp_util
  evaluation_id: ev_haiku_travel_srp
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.657'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_travel_spot_asr
  evaluation_id: ev_haiku_travel_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_travel_spot_util
  evaluation_id: ev_haiku_travel_spot
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.536'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_workspace_none_asr
  evaluation_id: ev_haiku_workspace_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_workspace_none_util
  evaluation_id: ev_haiku_workspace_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.857'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_workspace_srp_asr
  evaluation_id: ev_haiku_workspace_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_workspace_srp_util
  evaluation_id: ev_haiku_workspace_srp
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.720'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_workspace_spot_asr
  evaluation_id: ev_haiku_workspace_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_haiku_workspace_spot_util
  evaluation_id: ev_haiku_workspace_spot
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.809'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_banking_none_asr
  evaluation_id: ev_sonnet_banking_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_banking_none_util
  evaluation_id: ev_sonnet_banking_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.799'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_banking_srp_asr
  evaluation_id: ev_sonnet_banking_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_banking_srp_util
  evaluation_id: ev_sonnet_banking_srp
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.604'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_banking_spot_asr
  evaluation_id: ev_sonnet_banking_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_banking_spot_util
  evaluation_id: ev_sonnet_banking_spot
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.812'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_slack_none_asr
  evaluation_id: ev_sonnet_slack_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_slack_none_util
  evaluation_id: ev_sonnet_slack_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.714'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_slack_srp_asr
  evaluation_id: ev_sonnet_slack_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_slack_srp_util
  evaluation_id: ev_sonnet_slack_srp
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.657'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_slack_spot_asr
  evaluation_id: ev_sonnet_slack_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_slack_spot_util
  evaluation_id: ev_sonnet_slack_spot
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.714'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_travel_none_asr
  evaluation_id: ev_sonnet_travel_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.007'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_travel_none_util
  evaluation_id: ev_sonnet_travel_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.757'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_travel_srp_asr
  evaluation_id: ev_sonnet_travel_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.014'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_travel_srp_util
  evaluation_id: ev_sonnet_travel_srp
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.686'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_travel_spot_asr
  evaluation_id: ev_sonnet_travel_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.021'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_travel_spot_util
  evaluation_id: ev_sonnet_travel_spot
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.629'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_workspace_none_asr
  evaluation_id: ev_sonnet_workspace_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_workspace_none_util
  evaluation_id: ev_sonnet_workspace_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.977'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_workspace_srp_asr
  evaluation_id: ev_sonnet_workspace_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_workspace_srp_util
  evaluation_id: ev_sonnet_workspace_srp
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.948'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_workspace_spot_asr
  evaluation_id: ev_sonnet_workspace_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_sonnet_workspace_spot_util
  evaluation_id: ev_sonnet_workspace_spot
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility
  value: '0.973'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
comparisons:
- id: c_llama_banking_srp
  left: ev_llama_banking_srp
  right: ev_llama_banking_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_llama_banking_spot
  left: ev_llama_banking_spot
  right: ev_llama_banking_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_llama_slack_srp
  left: ev_llama_slack_srp
  right: ev_llama_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_llama_slack_spot
  left: ev_llama_slack_spot
  right: ev_llama_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_llama_travel_srp
  left: ev_llama_travel_srp
  right: ev_llama_travel_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_llama_travel_spot
  left: ev_llama_travel_spot
  right: ev_llama_travel_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_llama_workspace_srp
  left: ev_llama_workspace_srp
  right: ev_llama_workspace_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_llama_workspace_spot
  left: ev_llama_workspace_spot
  right: ev_llama_workspace_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_ds_banking_srp
  left: ev_ds_banking_srp
  right: ev_ds_banking_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_ds_banking_spot
  left: ev_ds_banking_spot
  right: ev_ds_banking_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_ds_slack_srp
  left: ev_ds_slack_srp
  right: ev_ds_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_ds_slack_spot
  left: ev_ds_slack_spot
  right: ev_ds_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_ds_travel_srp
  left: ev_ds_travel_srp
  right: ev_ds_travel_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_ds_travel_spot
  left: ev_ds_travel_spot
  right: ev_ds_travel_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_ds_workspace_srp
  left: ev_ds_workspace_srp
  right: ev_ds_workspace_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_ds_workspace_spot
  left: ev_ds_workspace_spot
  right: ev_ds_workspace_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_qwen_banking_srp
  left: ev_qwen_banking_srp
  right: ev_qwen_banking_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_qwen_banking_spot
  left: ev_qwen_banking_spot
  right: ev_qwen_banking_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_qwen_slack_srp
  left: ev_qwen_slack_srp
  right: ev_qwen_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_qwen_slack_spot
  left: ev_qwen_slack_spot
  right: ev_qwen_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_qwen_travel_srp
  left: ev_qwen_travel_srp
  right: ev_qwen_travel_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_qwen_travel_spot
  left: ev_qwen_travel_spot
  right: ev_qwen_travel_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_qwen_workspace_srp
  left: ev_qwen_workspace_srp
  right: ev_qwen_workspace_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_qwen_workspace_spot
  left: ev_qwen_workspace_spot
  right: ev_qwen_workspace_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_haiku_banking_srp
  left: ev_haiku_banking_srp
  right: ev_haiku_banking_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_haiku_banking_spot
  left: ev_haiku_banking_spot
  right: ev_haiku_banking_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_haiku_slack_srp
  left: ev_haiku_slack_srp
  right: ev_haiku_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_haiku_slack_spot
  left: ev_haiku_slack_spot
  right: ev_haiku_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_haiku_travel_srp
  left: ev_haiku_travel_srp
  right: ev_haiku_travel_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_haiku_travel_spot
  left: ev_haiku_travel_spot
  right: ev_haiku_travel_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_haiku_workspace_srp
  left: ev_haiku_workspace_srp
  right: ev_haiku_workspace_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_haiku_workspace_spot
  left: ev_haiku_workspace_spot
  right: ev_haiku_workspace_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_sonnet_banking_srp
  left: ev_sonnet_banking_srp
  right: ev_sonnet_banking_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_sonnet_banking_spot
  left: ev_sonnet_banking_spot
  right: ev_sonnet_banking_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_sonnet_slack_srp
  left: ev_sonnet_slack_srp
  right: ev_sonnet_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_sonnet_slack_spot
  left: ev_sonnet_slack_spot
  right: ev_sonnet_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_sonnet_travel_srp
  left: ev_sonnet_travel_srp
  right: ev_sonnet_travel_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_sonnet_travel_spot
  left: ev_sonnet_travel_spot
  right: ev_sonnet_travel_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_sonnet_workspace_srp
  left: ev_sonnet_workspace_srp
  right: ev_sonnet_workspace_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_sonnet_workspace_spot
  left: ev_sonnet_workspace_spot
  right: ev_sonnet_workspace_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
claims:
- id: k1
  text: the same defence sharply reduces attack success on one model and raises it on another (down 86.7 points, up 31.2 points under the same static attack)
  location: abstract
  anchor: the same defense sharply reduces attack success on one model yet increases it
  jobs:
  - R1
  supported_by:
  - c_ds_slack_srp
  - c_llama_banking_srp
  - c_llama_slack_srp
  - c_llama_banking_spot
  - c_llama_slack_spot
  - c_llama_travel_srp
  - c_llama_travel_spot
  - c_llama_workspace_srp
  - c_llama_workspace_spot
  - c_ds_banking_srp
  - c_ds_banking_spot
  - c_ds_slack_spot
  - c_ds_travel_srp
  - c_ds_travel_spot
  - c_ds_workspace_srp
  - c_ds_workspace_spot
  - c_qwen_banking_srp
  - c_qwen_banking_spot
  - c_qwen_slack_srp
  - c_qwen_slack_spot
  - c_qwen_travel_srp
  - c_qwen_travel_spot
  - c_qwen_workspace_srp
  - c_qwen_workspace_spot
  - c_haiku_banking_srp
  - c_haiku_banking_spot
  - c_haiku_slack_srp
  - c_haiku_slack_spot
  - c_haiku_travel_srp
  - c_haiku_travel_spot
  - c_haiku_workspace_srp
  - c_haiku_workspace_spot
  - c_sonnet_banking_srp
  - c_sonnet_banking_spot
  - c_sonnet_slack_srp
  - c_sonnet_slack_spot
  - c_sonnet_travel_srp
  - c_sonnet_travel_spot
  - c_sonnet_workspace_srp
  - c_sonnet_workspace_spot
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 6
    R2attack: Table 6 (utility in the static-attack runs)
    R2clean: not reported
schema_gaps:
- where: report
  note: the two Claude models show 0.0 ASR in almost every cell undefended, so their comparisons carry no security information; three models were logged under a shared registry alias, which the authors argue is constant across
    them (Appendix B); model names are as printed

`socportab_zenodo2026` (doi_10_5281_zenodo_20772702)

In [ ]:
%%writefile reports/records/socportab_zenodo2026.yaml
report:
  id: socportab_zenodo2026
  slug: doi_10_5281_zenodo_20772702
  title: 'Same Attack, Different Fates: Cross-Family Portability Failure of Prompt-Injection Defenses in LLM-Powered SOC Triage'
  first_author: Ankit Singh
  venue: Zenodo preprint (formatted for AISec 2026 with a placeholder DOI)
  peer_reviewed: 'no'
  doi: 10.5281/zenodo.20772702
  version_read: Zenodo version (20 June 2026)
  first_date: '2026-06-20'
  read: F
  role: re_measures
  code_release: analysis scripts named (stats_to_9.py); not checked
  extraction_scope: 'Section 7: Table 8 rows Unprotected, System Warning, XML Delimiters and Spotlighting for the five scorable open-weight models; the abstract claim (Allowlist Sanitization, a sanitiser, Dynamic Canary and
    the LLM Classifier, which detect, and the Stacked combination are left out; the frontier validation tests no defence)'
defences:
- id: warn_gemma
  name: System Warning (prompt-level warning wrapper)
  origin: other_report
  base_checkpoint: Gemma 2 9B (q4_0)
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: 'SOC triage prompt: alert telemetry with attacker-controlled fields, structured JSON decision (escalate or close)'
  layout_compat: &id001
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: We evaluate seven lightweight defenses applied as prompt-level wrappers
- id: xml_gemma
  name: XML Delimiters around the telemetry
  origin: other_report
  base_checkpoint: Gemma 2 9B (q4_0)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: 'SOC triage prompt: alert telemetry with attacker-controlled fields, structured JSON decision (escalate or close)'
  layout_compat: *id001
  anchor: We evaluate seven lightweight defenses applied as prompt-level wrappers
- id: spot_gemma
  name: Spotlighting (variant not named)
  origin: other_report
  base_checkpoint: Gemma 2 9B (q4_0)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: 'SOC triage prompt: alert telemetry with attacker-controlled fields, structured JSON decision (escalate or close)'
  layout_compat: *id001
  anchor: We evaluate seven lightweight defenses applied as prompt-level wrappers
- id: warn_llama
  name: System Warning (prompt-level warning wrapper)
  origin: other_report
  base_checkpoint: Llama 3 8B (q4_0)
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: 'SOC triage prompt: alert telemetry with attacker-controlled fields, structured JSON decision (escalate or close)'
  layout_compat: *id001
  anchor: We evaluate seven lightweight defenses applied as prompt-level wrappers
- id: xml_llama
  name: XML Delimiters around the telemetry
  origin: other_report
  base_checkpoint: Llama 3 8B (q4_0)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: 'SOC triage prompt: alert telemetry with attacker-controlled fields, structured JSON decision (escalate or close)'
  layout_compat: *id001
  anchor: We evaluate seven lightweight defenses applied as prompt-level wrappers
- id: spot_llama
  name: Spotlighting (variant not named)
  origin: other_report
  base_checkpoint: Llama 3 8B (q4_0)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: 'SOC triage prompt: alert telemetry with attacker-controlled fields, structured JSON decision (escalate or close)'
  layout_compat: *id001
  anchor: We evaluate seven lightweight defenses applied as prompt-level wrappers
- id: warn_phi
  name: System Warning (prompt-level warning wrapper)
  origin: other_report
  base_checkpoint: Phi-3 3.8B (q4_0)
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: 'SOC triage prompt: alert telemetry with attacker-controlled fields, structured JSON decision (escalate or close)'
  layout_compat: *id001
  anchor: We evaluate seven lightweight defenses applied as prompt-level wrappers
- id: xml_phi
  name: XML Delimiters around the telemetry
  origin: other_report
  base_checkpoint: Phi-3 3.8B (q4_0)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: 'SOC triage prompt: alert telemetry with attacker-controlled fields, structured JSON decision (escalate or close)'
  layout_compat: *id001
  anchor: We evaluate seven lightweight defenses applied as prompt-level wrappers
- id: spot_phi
  name: Spotlighting (variant not named)
  origin: other_report
  base_checkpoint: Phi-3 3.8B (q4_0)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: 'SOC triage prompt: alert telemetry with attacker-controlled fields, structured JSON decision (escalate or close)'
  layout_compat: *id001
  anchor: We evaluate seven lightweight defenses applied as prompt-level wrappers
- id: warn_mistral
  name: System Warning (prompt-level warning wrapper)
  origin: other_report
  base_checkpoint: Mistral 7B (q4_0)
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: 'SOC triage prompt: alert telemetry with attacker-controlled fields, structured JSON decision (escalate or close)'
  layout_compat: *id001
  anchor: We evaluate seven lightweight defenses applied as prompt-level wrappers
- id: xml_mistral
  name: XML Delimiters around the telemetry
  origin: other_report
  base_checkpoint: Mistral 7B (q4_0)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: 'SOC triage prompt: alert telemetry with attacker-controlled fields, structured JSON decision (escalate or close)'
  layout_compat: *id001
  anchor: We evaluate seven lightweight defenses applied as prompt-level wrappers
- id: spot_mistral
  name: Spotlighting (variant not named)
  origin: other_report
  base_checkpoint: Mistral 7B (q4_0)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: 'SOC triage prompt: alert telemetry with attacker-controlled fields, structured JSON decision (escalate or close)'
  layout_compat: *id001
  anchor: We evaluate seven lightweight defenses applied as prompt-level wrappers
- id: warn_qwen
  name: System Warning (prompt-level warning wrapper)
  origin: other_report
  base_checkpoint: Qwen 2.5 7B (q4_0)
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: 'SOC triage prompt: alert telemetry with attacker-controlled fields, structured JSON decision (escalate or close)'
  layout_compat: *id001
  anchor: We evaluate seven lightweight defenses applied as prompt-level wrappers
- id: xml_qwen
  name: XML Delimiters around the telemetry
  origin: other_report
  base_checkpoint: Qwen 2.5 7B (q4_0)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: 'SOC triage prompt: alert telemetry with attacker-controlled fields, structured JSON decision (escalate or close)'
  layout_compat: *id001
  anchor: We evaluate seven lightweight defenses applied as prompt-level wrappers
- id: spot_qwen
  name: Spotlighting (variant not named)
  origin: other_report
  base_checkpoint: Qwen 2.5 7B (q4_0)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: 'SOC triage prompt: alert telemetry with attacker-controlled fields, structured JSON decision (escalate or close)'
  layout_compat: *id001
  anchor: We evaluate seven lightweight defenses applied as prompt-level wrappers
evaluations:
- id: ev_gemma_none
  defence_id: none:Gemma 2 9B (q4_0)
  benchmark: 73 SOC alerts (28 malicious), 364 injected payloads in attacker-controlled telemetry fields
  attacker_type: static
  attacker_access: black_box
  judge: 'adversarial false closure: a malicious alert with an injected payload is closed or suppressed; unparseable outputs are not counted'
  layout_same_as_defence: 'yes'
  measured_by: socportab_zenodo2026
  page: 6
  decoding: not fully specified (Appendix D)
  sample_size: '364'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_llama_none
  defence_id: none:Llama 3 8B (q4_0)
  benchmark: 73 SOC alerts (28 malicious), 364 injected payloads in attacker-controlled telemetry fields
  attacker_type: static
  attacker_access: black_box
  judge: 'adversarial false closure: a malicious alert with an injected payload is closed or suppressed; unparseable outputs are not counted'
  layout_same_as_defence: 'yes'
  measured_by: socportab_zenodo2026
  page: 6
  decoding: not fully specified (Appendix D)
  sample_size: '364'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_phi_none
  defence_id: none:Phi-3 3.8B (q4_0)
  benchmark: 73 SOC alerts (28 malicious), 364 injected payloads in attacker-controlled telemetry fields
  attacker_type: static
  attacker_access: black_box
  judge: 'adversarial false closure: a malicious alert with an injected payload is closed or suppressed; unparseable outputs are not counted'
  layout_same_as_defence: 'yes'
  measured_by: socportab_zenodo2026
  page: 6
  decoding: not fully specified (Appendix D)
  sample_size: '364'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_mistral_none
  defence_id: none:Mistral 7B (q4_0)
  benchmark: 73 SOC alerts (28 malicious), 364 injected payloads in attacker-controlled telemetry fields
  attacker_type: static
  attacker_access: black_box
  judge: 'adversarial false closure: a malicious alert with an injected payload is closed or suppressed; unparseable outputs are not counted'
  layout_same_as_defence: 'yes'
  measured_by: socportab_zenodo2026
  page: 6
  decoding: not fully specified (Appendix D)
  sample_size: '364'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_qwen_none
  defence_id: none:Qwen 2.5 7B (q4_0)
  benchmark: 73 SOC alerts (28 malicious), 364 injected payloads in attacker-controlled telemetry fields
  attacker_type: static
  attacker_access: black_box
  judge: 'adversarial false closure: a malicious alert with an injected payload is closed or suppressed; unparseable outputs are not counted'
  layout_same_as_defence: 'yes'
  measured_by: socportab_zenodo2026
  page: 6
  decoding: not fully specified (Appendix D)
  sample_size: '364'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_gemma_warn
  defence_id: warn_gemma
  benchmark: 73 SOC alerts (28 malicious), 364 injected payloads in attacker-controlled telemetry fields
  attacker_type: static
  attacker_access: black_box
  judge: 'adversarial false closure: a malicious alert with an injected payload is closed or suppressed; unparseable outputs are not counted'
  layout_same_as_defence: 'yes'
  measured_by: socportab_zenodo2026
  page: 6
  decoding: not fully specified (Appendix D)
  sample_size: '364'
  pages:
    benchmark: 3
- id: ev_llama_warn
  defence_id: warn_llama
  benchmark: 73 SOC alerts (28 malicious), 364 injected payloads in attacker-controlled telemetry fields
  attacker_type: static
  attacker_access: black_box
  judge: 'adversarial false closure: a malicious alert with an injected payload is closed or suppressed; unparseable outputs are not counted'
  layout_same_as_defence: 'yes'
  measured_by: socportab_zenodo2026
  page: 6
  decoding: not fully specified (Appendix D)
  sample_size: '364'
  pages:
    benchmark: 3
- id: ev_phi_warn
  defence_id: warn_phi
  benchmark: 73 SOC alerts (28 malicious), 364 injected payloads in attacker-controlled telemetry fields
  attacker_type: static
  attacker_access: black_box
  judge: 'adversarial false closure: a malicious alert with an injected payload is closed or suppressed; unparseable outputs are not counted'
  layout_same_as_defence: 'yes'
  measured_by: socportab_zenodo2026
  page: 6
  decoding: not fully specified (Appendix D)
  sample_size: '364'
  pages:
    benchmark: 3
- id: ev_mistral_warn
  defence_id: warn_mistral
  benchmark: 73 SOC alerts (28 malicious), 364 injected payloads in attacker-controlled telemetry fields
  attacker_type: static
  attacker_access: black_box
  judge: 'adversarial false closure: a malicious alert with an injected payload is closed or suppressed; unparseable outputs are not counted'
  layout_same_as_defence: 'yes'
  measured_by: socportab_zenodo2026
  page: 6
  decoding: not fully specified (Appendix D)
  sample_size: '364'
  pages:
    benchmark: 3
- id: ev_qwen_warn
  defence_id: warn_qwen
  benchmark: 73 SOC alerts (28 malicious), 364 injected payloads in attacker-controlled telemetry fields
  attacker_type: static
  attacker_access: black_box
  judge: 'adversarial false closure: a malicious alert with an injected payload is closed or suppressed; unparseable outputs are not counted'
  layout_same_as_defence: 'yes'
  measured_by: socportab_zenodo2026
  page: 6
  decoding: not fully specified (Appendix D)
  sample_size: '364'
  pages:
    benchmark: 3
- id: ev_gemma_xml
  defence_id: xml_gemma
  benchmark: 73 SOC alerts (28 malicious), 364 injected payloads in attacker-controlled telemetry fields
  attacker_type: static
  attacker_access: black_box
  judge: 'adversarial false closure: a malicious alert with an injected payload is closed or suppressed; unparseable outputs are not counted'
  layout_same_as_defence: 'yes'
  measured_by: socportab_zenodo2026
  page: 6
  decoding: not fully specified (Appendix D)
  sample_size: '364'
  pages:
    benchmark: 3
- id: ev_llama_xml
  defence_id: xml_llama
  benchmark: 73 SOC alerts (28 malicious), 364 injected payloads in attacker-controlled telemetry fields
  attacker_type: static
  attacker_access: black_box
  judge: 'adversarial false closure: a malicious alert with an injected payload is closed or suppressed; unparseable outputs are not counted'
  layout_same_as_defence: 'yes'
  measured_by: socportab_zenodo2026
  page: 6
  decoding: not fully specified (Appendix D)
  sample_size: '364'
  pages:
    benchmark: 3
- id: ev_phi_xml
  defence_id: xml_phi
  benchmark: 73 SOC alerts (28 malicious), 364 injected payloads in attacker-controlled telemetry fields
  attacker_type: static
  attacker_access: black_box
  judge: 'adversarial false closure: a malicious alert with an injected payload is closed or suppressed; unparseable outputs are not counted'
  layout_same_as_defence: 'yes'
  measured_by: socportab_zenodo2026
  page: 6
  decoding: not fully specified (Appendix D)
  sample_size: '364'
  pages:
    benchmark: 3
- id: ev_mistral_xml
  defence_id: xml_mistral
  benchmark: 73 SOC alerts (28 malicious), 364 injected payloads in attacker-controlled telemetry fields
  attacker_type: static
  attacker_access: black_box
  judge: 'adversarial false closure: a malicious alert with an injected payload is closed or suppressed; unparseable outputs are not counted'
  layout_same_as_defence: 'yes'
  measured_by: socportab_zenodo2026
  page: 6
  decoding: not fully specified (Appendix D)
  sample_size: '364'
  pages:
    benchmark: 3
- id: ev_qwen_xml
  defence_id: xml_qwen
  benchmark: 73 SOC alerts (28 malicious), 364 injected payloads in attacker-controlled telemetry fields
  attacker_type: static
  attacker_access: black_box
  judge: 'adversarial false closure: a malicious alert with an injected payload is closed or suppressed; unparseable outputs are not counted'
  layout_same_as_defence: 'yes'
  measured_by: socportab_zenodo2026
  page: 6
  decoding: not fully specified (Appendix D)
  sample_size: '364'
  pages:
    benchmark: 3
- id: ev_gemma_spot
  defence_id: spot_gemma
  benchmark: 73 SOC alerts (28 malicious), 364 injected payloads in attacker-controlled telemetry fields
  attacker_type: static
  attacker_access: black_box
  judge: 'adversarial false closure: a malicious alert with an injected payload is closed or suppressed; unparseable outputs are not counted'
  layout_same_as_defence: 'yes'
  measured_by: socportab_zenodo2026
  page: 6
  decoding: not fully specified (Appendix D)
  sample_size: '364'
  pages:
    benchmark: 3
- id: ev_llama_spot
  defence_id: spot_llama
  benchmark: 73 SOC alerts (28 malicious), 364 injected payloads in attacker-controlled telemetry fields
  attacker_type: static
  attacker_access: black_box
  judge: 'adversarial false closure: a malicious alert with an injected payload is closed or suppressed; unparseable outputs are not counted'
  layout_same_as_defence: 'yes'
  measured_by: socportab_zenodo2026
  page: 6
  decoding: not fully specified (Appendix D)
  sample_size: '364'
  pages:
    benchmark: 3
- id: ev_phi_spot
  defence_id: spot_phi
  benchmark: 73 SOC alerts (28 malicious), 364 injected payloads in attacker-controlled telemetry fields
  attacker_type: static
  attacker_access: black_box
  judge: 'adversarial false closure: a malicious alert with an injected payload is closed or suppressed; unparseable outputs are not counted'
  layout_same_as_defence: 'yes'
  measured_by: socportab_zenodo2026
  page: 6
  decoding: not fully specified (Appendix D)
  sample_size: '364'
  pages:
    benchmark: 3
- id: ev_mistral_spot
  defence_id: spot_mistral
  benchmark: 73 SOC alerts (28 malicious), 364 injected payloads in attacker-controlled telemetry fields
  attacker_type: static
  attacker_access: black_box
  judge: 'adversarial false closure: a malicious alert with an injected payload is closed or suppressed; unparseable outputs are not counted'
  layout_same_as_defence: 'yes'
  measured_by: socportab_zenodo2026
  page: 6
  decoding: not fully specified (Appendix D)
  sample_size: '364'
  pages:
    benchmark: 3
- id: ev_qwen_spot
  defence_id: spot_qwen
  benchmark: 73 SOC alerts (28 malicious), 364 injected payloads in attacker-controlled telemetry fields
  attacker_type: static
  attacker_access: black_box
  judge: 'adversarial false closure: a malicious alert with an injected payload is closed or suppressed; unparseable outputs are not counted'
  layout_same_as_defence: 'yes'
  measured_by: socportab_zenodo2026
  page: 6
  decoding: not fully specified (Appendix D)
  sample_size: '364'
  pages:
    benchmark: 3
results:
- id: r_gemma_none
  evaluation_id: ev_gemma_none
  quantity: R1
  coverage: direct
  metric_as_printed: adversarial false-closure rate (AFCR)
  value: '37.64'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 8
- id: r_llama_none
  evaluation_id: ev_llama_none
  quantity: R1
  coverage: direct
  metric_as_printed: adversarial false-closure rate (AFCR)
  value: '53.57'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 8
- id: r_phi_none
  evaluation_id: ev_phi_none
  quantity: R1
  coverage: direct
  metric_as_printed: adversarial false-closure rate (AFCR)
  value: '68.96'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 8
- id: r_mistral_none
  evaluation_id: ev_mistral_none
  quantity: R1
  coverage: direct
  metric_as_printed: adversarial false-closure rate (AFCR)
  value: '39.01'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 8
- id: r_qwen_none
  evaluation_id: ev_qwen_none
  quantity: R1
  coverage: direct
  metric_as_printed: adversarial false-closure rate (AFCR)
  value: '100.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 8
- id: r_gemma_warn
  evaluation_id: ev_gemma_warn
  quantity: R1
  coverage: direct
  metric_as_printed: adversarial false-closure rate (AFCR)
  value: '30.49'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 8
- id: r_llama_warn
  evaluation_id: ev_llama_warn
  quantity: R1
  coverage: direct
  metric_as_printed: adversarial false-closure rate (AFCR)
  value: '85.44'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 8
- id: r_phi_warn
  evaluation_id: ev_phi_warn
  quantity: R1
  coverage: direct
  metric_as_printed: adversarial false-closure rate (AFCR)
  value: '70.33'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 8
- id: r_mistral_warn
  evaluation_id: ev_mistral_warn
  quantity: R1
  coverage: direct
  metric_as_printed: adversarial false-closure rate (AFCR)
  value: '35.71'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 8
- id: r_qwen_warn
  evaluation_id: ev_qwen_warn
  quantity: R1
  coverage: direct
  metric_as_printed: adversarial false-closure rate (AFCR)
  value: '100.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 8
- id: r_gemma_xml
  evaluation_id: ev_gemma_xml
  quantity: R1
  coverage: direct
  metric_as_printed: adversarial false-closure rate (AFCR)
  value: '26.10'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 8
- id: r_llama_xml
  evaluation_id: ev_llama_xml
  quantity: R1
  coverage: direct
  metric_as_printed: adversarial false-closure rate (AFCR)
  value: '45.05'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 8
- id: r_phi_xml
  evaluation_id: ev_phi_xml
  quantity: R1
  coverage: direct
  metric_as_printed: adversarial false-closure rate (AFCR)
  value: '63.74'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 8
- id: r_mistral_xml
  evaluation_id: ev_mistral_xml
  quantity: R1
  coverage: direct
  metric_as_printed: adversarial false-closure rate (AFCR)
  value: '27.47'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 8
- id: r_qwen_xml
  evaluation_id: ev_qwen_xml
  quantity: R1
  coverage: direct
  metric_as_printed: adversarial false-closure rate (AFCR)
  value: '100.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 8
- id: r_gemma_spot
  evaluation_id: ev_gemma_spot
  quantity: R1
  coverage: direct
  metric_as_printed: adversarial false-closure rate (AFCR)
  value: '34.62'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 8
- id: r_llama_spot
  evaluation_id: ev_llama_spot
  quantity: R1
  coverage: direct
  metric_as_printed: adversarial false-closure rate (AFCR)
  value: '48.08'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 8
- id: r_phi_spot
  evaluation_id: ev_phi_spot
  quantity: R1
  coverage: direct
  metric_as_printed: adversarial false-closure rate (AFCR)
  value: '68.96'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 8
- id: r_mistral_spot
  evaluation_id: ev_mistral_spot
  quantity: R1
  coverage: direct
  metric_as_printed: adversarial false-closure rate (AFCR)
  value: '39.01'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 8
- id: r_qwen_spot
  evaluation_id: ev_qwen_spot
  quantity: R1
  coverage: direct
  metric_as_printed: adversarial false-closure rate (AFCR)
  value: '100.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 8
comparisons:
- id: c_gemma_warn
  left: ev_gemma_warn
  right: ev_gemma_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_gemma_xml
  left: ev_gemma_xml
  right: ev_gemma_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_gemma_spot
  left: ev_gemma_spot
  right: ev_gemma_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_llama_warn
  left: ev_llama_warn
  right: ev_llama_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_llama_xml
  left: ev_llama_xml
  right: ev_llama_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_llama_spot
  left: ev_llama_spot
  right: ev_llama_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_phi_warn
  left: ev_phi_warn
  right: ev_phi_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_phi_xml
  left: ev_phi_xml
  right: ev_phi_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_phi_spot
  left: ev_phi_spot
  right: ev_phi_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_mistral_warn
  left: ev_mistral_warn
  right: ev_mistral_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_mistral_xml
  left: ev_mistral_xml
  right: ev_mistral_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_mistral_spot
  left: ev_mistral_spot
  right: ev_mistral_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_qwen_warn
  left: ev_qwen_warn
  right: ev_qwen_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_qwen_xml
  left: ev_qwen_xml
  right: ev_qwen_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_qwen_spot
  left: ev_qwen_spot
  right: ev_qwen_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
claims:
- id: k1
  text: 'defences are not portable across model families: one that lowers AFCR on one model can leave another unchanged or make it substantially worse'
  location: abstract
  anchor: defenses are not portable across families
  jobs:
  - R1
  supported_by:
  - c_gemma_warn
  - c_gemma_xml
  - c_gemma_spot
  - c_llama_warn
  - c_llama_xml
  - c_llama_spot
  - c_phi_warn
  - c_phi_xml
  - c_phi_spot
  - c_mistral_warn
  - c_mistral_xml
  - c_mistral_spot
  - c_qwen_warn
  - c_qwen_xml
  - c_qwen_spot
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 8 (AFCR)
    R2clean: benign action accuracy is reported only without defences (Table 6)
    R2attack: not reported
schema_gaps:
- where: report
  note: AFCR is the attack-success measure here (false closure of malicious alerts); Qwen 2.5 never escalates any alert (0% intervention, 0% action accuracy), so its AFCR is 100% under every condition and its comparisons are
    uninformative; decoding settings are not fully specified, as the paper states

`transfact_zenodo2026` (doi_10_5281_zenodo_21120124)

In [ ]:
%%writefile reports/records/transfact_zenodo2026.yaml
report:
  id: transfact_zenodo2026
  slug: doi_10_5281_zenodo_21120124
  title: 'Prompt Transformation Effects Do Not Transfer Across Benchmarks or Model Families: A Factorial Study of Defense Evaluation in LLM Agents'
  first_author: Munish Singh
  venue: Zenodo preprint
  peer_reviewed: 'no'
  doi: 10.5281/zenodo.21120124
  version_read: Zenodo version (2 July 2026)
  first_date: '2026-07-02'
  read: F
  role: re_measures
  code_release: not stated
  extraction_scope: 'Appendix A: Table 1 (all 65 cells: three AgentDojo suites and BIPIA email, five models, undefended, SRP, Spotlighting and the neutral-preamble control); the abstract claims (Table 2 per-injection-task
    breakdown and Figure 1 not coded separately)'
defences:
- id: srp_haiku
  name: Self-Reflective Prompting (structured verification block prepended to each input)
  origin: other_report
  base_checkpoint: Claude Haiku 4.5
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: AgentDojo tool-calling agent (injection in tool outputs) or BIPIA email question answering (injection in the email)
  layout_compat: &id001
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: prepends a structured verification block to each
- id: spot_haiku
  name: Spotlighting by datamarking (spaces replaced with U+02C8, system preamble)
  origin: other_report
  base_checkpoint: Claude Haiku 4.5
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: AgentDojo tool-calling agent (injection in tool outputs) or BIPIA email question answering (injection in the email)
  layout_compat: *id001
  anchor: The datamarking variant used here substitutes every space character
- id: srp_sonnet
  name: Self-Reflective Prompting (structured verification block prepended to each input)
  origin: other_report
  base_checkpoint: Claude Sonnet 4.6
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: AgentDojo tool-calling agent (injection in tool outputs) or BIPIA email question answering (injection in the email)
  layout_compat: *id001
  anchor: prepends a structured verification block to each
- id: spot_sonnet
  name: Spotlighting by datamarking (spaces replaced with U+02C8, system preamble)
  origin: other_report
  base_checkpoint: Claude Sonnet 4.6
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: AgentDojo tool-calling agent (injection in tool outputs) or BIPIA email question answering (injection in the email)
  layout_compat: *id001
  anchor: The datamarking variant used here substitutes every space character
- id: srp_llama
  name: Self-Reflective Prompting (structured verification block prepended to each input)
  origin: other_report
  base_checkpoint: Llama 4 Scout
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: AgentDojo tool-calling agent (injection in tool outputs) or BIPIA email question answering (injection in the email)
  layout_compat: *id001
  anchor: prepends a structured verification block to each
- id: spot_llama
  name: Spotlighting by datamarking (spaces replaced with U+02C8, system preamble)
  origin: other_report
  base_checkpoint: Llama 4 Scout
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: AgentDojo tool-calling agent (injection in tool outputs) or BIPIA email question answering (injection in the email)
  layout_compat: *id001
  anchor: The datamarking variant used here substitutes every space character
- id: neut_llama
  name: length-matched neutral preamble (placebo control with the structure of SRP and unrelated content)
  origin: this_report
  base_checkpoint: Llama 4 Scout
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: AgentDojo tool-calling agent (injection in tool outputs) or BIPIA email question answering (injection in the email)
  layout_compat: *id001
  anchor: The neutral preamble matches SRP in length
- id: srp_ds
  name: Self-Reflective Prompting (structured verification block prepended to each input)
  origin: other_report
  base_checkpoint: DeepSeek V3.2
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: AgentDojo tool-calling agent (injection in tool outputs) or BIPIA email question answering (injection in the email)
  layout_compat: *id001
  anchor: prepends a structured verification block to each
- id: spot_ds
  name: Spotlighting by datamarking (spaces replaced with U+02C8, system preamble)
  origin: other_report
  base_checkpoint: DeepSeek V3.2
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: AgentDojo tool-calling agent (injection in tool outputs) or BIPIA email question answering (injection in the email)
  layout_compat: *id001
  anchor: The datamarking variant used here substitutes every space character
- id: neut_ds
  name: length-matched neutral preamble (placebo control with the structure of SRP and unrelated content)
  origin: this_report
  base_checkpoint: DeepSeek V3.2
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: AgentDojo tool-calling agent (injection in tool outputs) or BIPIA email question answering (injection in the email)
  layout_compat: *id001
  anchor: The neutral preamble matches SRP in length
- id: srp_qwen
  name: Self-Reflective Prompting (structured verification block prepended to each input)
  origin: other_report
  base_checkpoint: Qwen 3 32B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: AgentDojo tool-calling agent (injection in tool outputs) or BIPIA email question answering (injection in the email)
  layout_compat: *id001
  anchor: prepends a structured verification block to each
- id: spot_qwen
  name: Spotlighting by datamarking (spaces replaced with U+02C8, system preamble)
  origin: other_report
  base_checkpoint: Qwen 3 32B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: AgentDojo tool-calling agent (injection in tool outputs) or BIPIA email question answering (injection in the email)
  layout_compat: *id001
  anchor: The datamarking variant used here substitutes every space character
- id: neut_qwen
  name: length-matched neutral preamble (placebo control with the structure of SRP and unrelated content)
  origin: this_report
  base_checkpoint: Qwen 3 32B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: AgentDojo tool-calling agent (injection in tool outputs) or BIPIA email question answering (injection in the email)
  layout_compat: *id001
  anchor: The neutral preamble matches SRP in length
evaluations:
- id: ev_haiku_banking_none
  defence_id: none:Claude Haiku 4.5
  benchmark: AgentDojo v1.2 banking suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 12
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_haiku_banking_srp
  defence_id: srp_haiku
  benchmark: AgentDojo v1.2 banking suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 12
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
- id: ev_haiku_banking_spot
  defence_id: spot_haiku
  benchmark: AgentDojo v1.2 banking suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 12
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
- id: ev_sonnet_banking_none
  defence_id: none:Claude Sonnet 4.6
  benchmark: AgentDojo v1.2 banking suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 12
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_sonnet_banking_srp
  defence_id: srp_sonnet
  benchmark: AgentDojo v1.2 banking suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 12
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
- id: ev_sonnet_banking_spot
  defence_id: spot_sonnet
  benchmark: AgentDojo v1.2 banking suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 12
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
- id: ev_llama_banking_none
  defence_id: none:Llama 4 Scout
  benchmark: AgentDojo v1.2 banking suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 12
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_llama_banking_srp
  defence_id: srp_llama
  benchmark: AgentDojo v1.2 banking suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 12
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
- id: ev_llama_banking_spot
  defence_id: spot_llama
  benchmark: AgentDojo v1.2 banking suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 12
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
- id: ev_llama_banking_neut
  defence_id: neut_llama
  benchmark: AgentDojo v1.2 banking suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 12
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
- id: ev_ds_banking_none
  defence_id: none:DeepSeek V3.2
  benchmark: AgentDojo v1.2 banking suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 12
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_ds_banking_srp
  defence_id: srp_ds
  benchmark: AgentDojo v1.2 banking suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 12
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
- id: ev_ds_banking_spot
  defence_id: spot_ds
  benchmark: AgentDojo v1.2 banking suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 12
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
- id: ev_ds_banking_neut
  defence_id: neut_ds
  benchmark: AgentDojo v1.2 banking suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 12
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
- id: ev_qwen_banking_none
  defence_id: none:Qwen 3 32B
  benchmark: AgentDojo v1.2 banking suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 12
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_qwen_banking_srp
  defence_id: srp_qwen
  benchmark: AgentDojo v1.2 banking suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 12
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
- id: ev_qwen_banking_spot
  defence_id: spot_qwen
  benchmark: AgentDojo v1.2 banking suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 12
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
- id: ev_qwen_banking_neut
  defence_id: neut_qwen
  benchmark: AgentDojo v1.2 banking suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 12
  decoding: not reported
  sample_size: '144'
  pages:
    benchmark: 3
- id: ev_haiku_slack_none
  defence_id: none:Claude Haiku 4.5
  benchmark: AgentDojo v1.2 slack suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 12
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_haiku_slack_srp
  defence_id: srp_haiku
  benchmark: AgentDojo v1.2 slack suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 12
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
- id: ev_haiku_slack_spot
  defence_id: spot_haiku
  benchmark: AgentDojo v1.2 slack suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 12
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
- id: ev_sonnet_slack_none
  defence_id: none:Claude Sonnet 4.6
  benchmark: AgentDojo v1.2 slack suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 12
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_sonnet_slack_srp
  defence_id: srp_sonnet
  benchmark: AgentDojo v1.2 slack suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 12
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
- id: ev_sonnet_slack_spot
  defence_id: spot_sonnet
  benchmark: AgentDojo v1.2 slack suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
- id: ev_llama_slack_none
  defence_id: none:Llama 4 Scout
  benchmark: AgentDojo v1.2 slack suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_llama_slack_srp
  defence_id: srp_llama
  benchmark: AgentDojo v1.2 slack suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
- id: ev_llama_slack_spot
  defence_id: spot_llama
  benchmark: AgentDojo v1.2 slack suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
- id: ev_llama_slack_neut
  defence_id: neut_llama
  benchmark: AgentDojo v1.2 slack suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
- id: ev_ds_slack_none
  defence_id: none:DeepSeek V3.2
  benchmark: AgentDojo v1.2 slack suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_ds_slack_srp
  defence_id: srp_ds
  benchmark: AgentDojo v1.2 slack suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
- id: ev_ds_slack_spot
  defence_id: spot_ds
  benchmark: AgentDojo v1.2 slack suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
- id: ev_qwen_slack_none
  defence_id: none:Qwen 3 32B
  benchmark: AgentDojo v1.2 slack suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_qwen_slack_srp
  defence_id: srp_qwen
  benchmark: AgentDojo v1.2 slack suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
- id: ev_qwen_slack_spot
  defence_id: spot_qwen
  benchmark: AgentDojo v1.2 slack suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '105'
  pages:
    benchmark: 3
- id: ev_haiku_travel_none
  defence_id: none:Claude Haiku 4.5
  benchmark: AgentDojo v1.2 travel suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_haiku_travel_srp
  defence_id: srp_haiku
  benchmark: AgentDojo v1.2 travel suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
- id: ev_haiku_travel_spot
  defence_id: spot_haiku
  benchmark: AgentDojo v1.2 travel suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
- id: ev_sonnet_travel_none
  defence_id: none:Claude Sonnet 4.6
  benchmark: AgentDojo v1.2 travel suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_sonnet_travel_srp
  defence_id: srp_sonnet
  benchmark: AgentDojo v1.2 travel suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
- id: ev_sonnet_travel_spot
  defence_id: spot_sonnet
  benchmark: AgentDojo v1.2 travel suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
- id: ev_llama_travel_none
  defence_id: none:Llama 4 Scout
  benchmark: AgentDojo v1.2 travel suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_llama_travel_srp
  defence_id: srp_llama
  benchmark: AgentDojo v1.2 travel suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
- id: ev_llama_travel_spot
  defence_id: spot_llama
  benchmark: AgentDojo v1.2 travel suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
- id: ev_llama_travel_neut
  defence_id: neut_llama
  benchmark: AgentDojo v1.2 travel suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
- id: ev_ds_travel_none
  defence_id: none:DeepSeek V3.2
  benchmark: AgentDojo v1.2 travel suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_ds_travel_srp
  defence_id: srp_ds
  benchmark: AgentDojo v1.2 travel suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
- id: ev_ds_travel_spot
  defence_id: spot_ds
  benchmark: AgentDojo v1.2 travel suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
- id: ev_qwen_travel_none
  defence_id: none:Qwen 3 32B
  benchmark: AgentDojo v1.2 travel suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_qwen_travel_srp
  defence_id: srp_qwen
  benchmark: AgentDojo v1.2 travel suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
- id: ev_qwen_travel_spot
  defence_id: spot_qwen
  benchmark: AgentDojo v1.2 travel suite
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo operational injection check
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '140'
  pages:
    benchmark: 3
- id: ev_haiku_email_none
  defence_id: none:Claude Haiku 4.5
  benchmark: BIPIA email subset (3,750 attack scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: LLM judge (not named)
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '3750'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_haiku_email_srp
  defence_id: srp_haiku
  benchmark: BIPIA email subset (3,750 attack scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: LLM judge (not named)
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '3750'
  pages:
    benchmark: 3
- id: ev_haiku_email_spot
  defence_id: spot_haiku
  benchmark: BIPIA email subset (3,750 attack scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: LLM judge (not named)
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '3750'
  pages:
    benchmark: 3
- id: ev_sonnet_email_none
  defence_id: none:Claude Sonnet 4.6
  benchmark: BIPIA email subset (3,750 attack scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: LLM judge (not named)
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '3750'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_sonnet_email_srp
  defence_id: srp_sonnet
  benchmark: BIPIA email subset (3,750 attack scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: LLM judge (not named)
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '3750'
  pages:
    benchmark: 3
- id: ev_sonnet_email_spot
  defence_id: spot_sonnet
  benchmark: BIPIA email subset (3,750 attack scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: LLM judge (not named)
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '3750'
  pages:
    benchmark: 3
- id: ev_llama_email_none
  defence_id: none:Llama 4 Scout
  benchmark: BIPIA email subset (3,750 attack scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: LLM judge (not named)
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '3750'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_llama_email_srp
  defence_id: srp_llama
  benchmark: BIPIA email subset (3,750 attack scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: LLM judge (not named)
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '3750'
  pages:
    benchmark: 3
- id: ev_llama_email_spot
  defence_id: spot_llama
  benchmark: BIPIA email subset (3,750 attack scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: LLM judge (not named)
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '3750'
  pages:
    benchmark: 3
- id: ev_ds_email_none
  defence_id: none:DeepSeek V3.2
  benchmark: BIPIA email subset (3,750 attack scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: LLM judge (not named)
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '3750'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_ds_email_srp
  defence_id: srp_ds
  benchmark: BIPIA email subset (3,750 attack scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: LLM judge (not named)
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '3750'
  pages:
    benchmark: 3
- id: ev_ds_email_spot
  defence_id: spot_ds
  benchmark: BIPIA email subset (3,750 attack scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: LLM judge (not named)
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '3750'
  pages:
    benchmark: 3
- id: ev_qwen_email_none
  defence_id: none:Qwen 3 32B
  benchmark: BIPIA email subset (3,750 attack scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: LLM judge (not named)
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '3750'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_qwen_email_srp
  defence_id: srp_qwen
  benchmark: BIPIA email subset (3,750 attack scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: LLM judge (not named)
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '3750'
  pages:
    benchmark: 3
- id: ev_qwen_email_spot
  defence_id: spot_qwen
  benchmark: BIPIA email subset (3,750 attack scenarios)
  attacker_type: static
  attacker_access: black_box
  judge: LLM judge (not named)
  layout_same_as_defence: 'yes'
  measured_by: transfact_zenodo2026
  page: 13
  decoding: not reported
  sample_size: '3750'
  pages:
    benchmark: 3
results:
- id: r_haiku_banking_none
  evaluation_id: ev_haiku_banking_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_haiku_banking_srp
  evaluation_id: ev_haiku_banking_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_haiku_banking_spot
  evaluation_id: ev_haiku_banking_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_sonnet_banking_none
  evaluation_id: ev_sonnet_banking_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_sonnet_banking_srp
  evaluation_id: ev_sonnet_banking_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_sonnet_banking_spot
  evaluation_id: ev_sonnet_banking_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_llama_banking_none
  evaluation_id: ev_llama_banking_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '30.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_llama_banking_srp
  evaluation_id: ev_llama_banking_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '71.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_llama_banking_spot
  evaluation_id: ev_llama_banking_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '22.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_llama_banking_neut
  evaluation_id: ev_llama_banking_neut
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '41.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_ds_banking_none
  evaluation_id: ev_ds_banking_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '71.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_ds_banking_srp
  evaluation_id: ev_ds_banking_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '2.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_ds_banking_spot
  evaluation_id: ev_ds_banking_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '72.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_ds_banking_neut
  evaluation_id: ev_ds_banking_neut
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '60.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_qwen_banking_none
  evaluation_id: ev_qwen_banking_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '37.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_qwen_banking_srp
  evaluation_id: ev_qwen_banking_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '38.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_qwen_banking_spot
  evaluation_id: ev_qwen_banking_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '41.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_qwen_banking_neut
  evaluation_id: ev_qwen_banking_neut
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '43.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_haiku_slack_none
  evaluation_id: ev_haiku_slack_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_haiku_slack_srp
  evaluation_id: ev_haiku_slack_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_haiku_slack_spot
  evaluation_id: ev_haiku_slack_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_sonnet_slack_none
  evaluation_id: ev_sonnet_slack_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_sonnet_slack_srp
  evaluation_id: ev_sonnet_slack_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_sonnet_slack_spot
  evaluation_id: ev_sonnet_slack_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_llama_slack_none
  evaluation_id: ev_llama_slack_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '31.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_llama_slack_srp
  evaluation_id: ev_llama_slack_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '59.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_llama_slack_spot
  evaluation_id: ev_llama_slack_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '24.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_llama_slack_neut
  evaluation_id: ev_llama_slack_neut
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '35.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_ds_slack_none
  evaluation_id: ev_ds_slack_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '96.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_ds_slack_srp
  evaluation_id: ev_ds_slack_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '8.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_ds_slack_spot
  evaluation_id: ev_ds_slack_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '94.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_qwen_slack_none
  evaluation_id: ev_qwen_slack_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '66.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_qwen_slack_srp
  evaluation_id: ev_qwen_slack_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '67.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_qwen_slack_spot
  evaluation_id: ev_qwen_slack_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '64.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_haiku_travel_none
  evaluation_id: ev_haiku_travel_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_haiku_travel_srp
  evaluation_id: ev_haiku_travel_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_haiku_travel_spot
  evaluation_id: ev_haiku_travel_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_sonnet_travel_none
  evaluation_id: ev_sonnet_travel_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_sonnet_travel_srp
  evaluation_id: ev_sonnet_travel_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_sonnet_travel_spot
  evaluation_id: ev_sonnet_travel_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '2.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_llama_travel_none
  evaluation_id: ev_llama_travel_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '15.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_llama_travel_srp
  evaluation_id: ev_llama_travel_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '21.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_llama_travel_spot
  evaluation_id: ev_llama_travel_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '7.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_llama_travel_neut
  evaluation_id: ev_llama_travel_neut
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '16.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_ds_travel_none
  evaluation_id: ev_ds_travel_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '75.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_ds_travel_srp
  evaluation_id: ev_ds_travel_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '2.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_ds_travel_spot
  evaluation_id: ev_ds_travel_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '72.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_qwen_travel_none
  evaluation_id: ev_qwen_travel_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '51.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_qwen_travel_srp
  evaluation_id: ev_qwen_travel_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '43.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_qwen_travel_spot
  evaluation_id: ev_qwen_travel_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '34.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_haiku_email_none
  evaluation_id: ev_haiku_email_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '43.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_haiku_email_srp
  evaluation_id: ev_haiku_email_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '22.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_haiku_email_spot
  evaluation_id: ev_haiku_email_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '13.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_sonnet_email_none
  evaluation_id: ev_sonnet_email_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '25.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_sonnet_email_srp
  evaluation_id: ev_sonnet_email_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '13.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_sonnet_email_spot
  evaluation_id: ev_sonnet_email_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '3.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_llama_email_none
  evaluation_id: ev_llama_email_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '42.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_llama_email_srp
  evaluation_id: ev_llama_email_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '40.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_llama_email_spot
  evaluation_id: ev_llama_email_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '20.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_ds_email_none
  evaluation_id: ev_ds_email_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '34.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_ds_email_srp
  evaluation_id: ev_ds_email_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '31.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_ds_email_spot
  evaluation_id: ev_ds_email_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '17.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_qwen_email_none
  evaluation_id: ev_qwen_email_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '26.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_qwen_email_srp
  evaluation_id: ev_qwen_email_srp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '22.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
- id: r_qwen_email_spot
  evaluation_id: ev_qwen_email_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '10.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 13
  location: Table 1
comparisons:
- id: c_haiku_banking_srp
  left: ev_haiku_banking_srp
  right: ev_haiku_banking_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_haiku_banking_spot
  left: ev_haiku_banking_spot
  right: ev_haiku_banking_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_sonnet_banking_srp
  left: ev_sonnet_banking_srp
  right: ev_sonnet_banking_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_sonnet_banking_spot
  left: ev_sonnet_banking_spot
  right: ev_sonnet_banking_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_llama_banking_srp
  left: ev_llama_banking_srp
  right: ev_llama_banking_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_llama_banking_spot
  left: ev_llama_banking_spot
  right: ev_llama_banking_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_llama_banking_neut
  left: ev_llama_banking_neut
  right: ev_llama_banking_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_llama_banking_srpneut
  left: ev_llama_banking_srp
  right: ev_llama_banking_neut
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: SRP's security content against a placebo block of the same length and structure
  page: 13
- id: c_ds_banking_srp
  left: ev_ds_banking_srp
  right: ev_ds_banking_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_ds_banking_spot
  left: ev_ds_banking_spot
  right: ev_ds_banking_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_ds_banking_neut
  left: ev_ds_banking_neut
  right: ev_ds_banking_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_ds_banking_srpneut
  left: ev_ds_banking_srp
  right: ev_ds_banking_neut
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: SRP's security content against a placebo block of the same length and structure
  page: 13
- id: c_qwen_banking_srp
  left: ev_qwen_banking_srp
  right: ev_qwen_banking_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_qwen_banking_spot
  left: ev_qwen_banking_spot
  right: ev_qwen_banking_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_qwen_banking_neut
  left: ev_qwen_banking_neut
  right: ev_qwen_banking_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_qwen_banking_srpneut
  left: ev_qwen_banking_srp
  right: ev_qwen_banking_neut
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: SRP's security content against a placebo block of the same length and structure
  page: 13
- id: c_haiku_slack_srp
  left: ev_haiku_slack_srp
  right: ev_haiku_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_haiku_slack_spot
  left: ev_haiku_slack_spot
  right: ev_haiku_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_sonnet_slack_srp
  left: ev_sonnet_slack_srp
  right: ev_sonnet_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_sonnet_slack_spot
  left: ev_sonnet_slack_spot
  right: ev_sonnet_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_llama_slack_srp
  left: ev_llama_slack_srp
  right: ev_llama_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_llama_slack_spot
  left: ev_llama_slack_spot
  right: ev_llama_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_llama_slack_neut
  left: ev_llama_slack_neut
  right: ev_llama_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_llama_slack_srpneut
  left: ev_llama_slack_srp
  right: ev_llama_slack_neut
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: SRP's security content against a placebo block of the same length and structure
  page: 13
- id: c_ds_slack_srp
  left: ev_ds_slack_srp
  right: ev_ds_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_ds_slack_spot
  left: ev_ds_slack_spot
  right: ev_ds_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_qwen_slack_srp
  left: ev_qwen_slack_srp
  right: ev_qwen_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_qwen_slack_spot
  left: ev_qwen_slack_spot
  right: ev_qwen_slack_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_haiku_travel_srp
  left: ev_haiku_travel_srp
  right: ev_haiku_travel_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_haiku_travel_spot
  left: ev_haiku_travel_spot
  right: ev_haiku_travel_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_sonnet_travel_srp
  left: ev_sonnet_travel_srp
  right: ev_sonnet_travel_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_sonnet_travel_spot
  left: ev_sonnet_travel_spot
  right: ev_sonnet_travel_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_llama_travel_srp
  left: ev_llama_travel_srp
  right: ev_llama_travel_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_llama_travel_spot
  left: ev_llama_travel_spot
  right: ev_llama_travel_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_llama_travel_neut
  left: ev_llama_travel_neut
  right: ev_llama_travel_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_llama_travel_srpneut
  left: ev_llama_travel_srp
  right: ev_llama_travel_neut
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: SRP's security content against a placebo block of the same length and structure
  page: 13
- id: c_ds_travel_srp
  left: ev_ds_travel_srp
  right: ev_ds_travel_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_ds_travel_spot
  left: ev_ds_travel_spot
  right: ev_ds_travel_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_qwen_travel_srp
  left: ev_qwen_travel_srp
  right: ev_qwen_travel_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_qwen_travel_spot
  left: ev_qwen_travel_spot
  right: ev_qwen_travel_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_haiku_email_srp
  left: ev_haiku_email_srp
  right: ev_haiku_email_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_haiku_email_spot
  left: ev_haiku_email_spot
  right: ev_haiku_email_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_sonnet_email_srp
  left: ev_sonnet_email_srp
  right: ev_sonnet_email_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_sonnet_email_spot
  left: ev_sonnet_email_spot
  right: ev_sonnet_email_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_llama_email_srp
  left: ev_llama_email_srp
  right: ev_llama_email_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_llama_email_spot
  left: ev_llama_email_spot
  right: ev_llama_email_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_ds_email_srp
  left: ev_ds_email_srp
  right: ev_ds_email_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_ds_email_spot
  left: ev_ds_email_spot
  right: ev_ds_email_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_qwen_email_srp
  left: ev_qwen_email_srp
  right: ev_qwen_email_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
- id: c_qwen_email_spot
  left: ev_qwen_email_spot
  right: ev_qwen_email_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 13
claims:
- id: k1
  text: SRP cuts DeepSeek V3.2's ASR by 69.5 points on AgentDojo Banking while raising Llama 4 Scout's by 41.0 points
  location: abstract
  anchor: Self-Reflective Prompting reduces DeepSeek V3.2’s attack success rate by 69.5
  jobs:
  - R1
  supported_by:
  - c_ds_banking_srp
  - c_llama_banking_srp
  - c_ds_slack_srp
  - c_llama_slack_srp
  page: 1
- id: k2
  text: the same defence behaves differently on agentic and document benchmarks
  location: abstract
  anchor: the same defense behaves qualitatively differently on agentic and document benchmarks
  jobs:
  - R1
  supported_by:
  - c_llama_email_srp
  - c_llama_banking_srp
  - c_haiku_email_srp
  - c_sonnet_email_srp
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 1
    R2clean: not reported
    R2attack: not reported
schema_gaps:
- where: report
  note: same authors as the defence-backfire study (zenodo.20735483) with overlapping models, suites and defences; the AgentDojo cells here differ from that report's (Llama Scout banking undefended 30.6 against 31.9), so they
    are separate runs; the two Claude models are near zero on AgentDojo throughout; model names are as printed

`smallagentsq_zenodo2026` (doi_10_5281_zenodo_21343930)

In [ ]:
%%writefile reports/records/smallagentsq_zenodo2026.yaml
report:
  id: smallagentsq_zenodo2026
  slug: doi_10_5281_zenodo_21343930
  title: What Determines the Vulnerability of Small LLM Agents to Indirect Prompt Injection? A Systematic Study of Quantization, Model Size, Architecture, and Attack Style
  first_author: Aleksandr Kuleshov
  venue: Zenodo preprint
  peer_reviewed: 'no'
  doi: 10.5281/zenodo.21343930
  version_read: Zenodo version
  first_date: '2026-07-13'
  read: F
  role: re_measures
  code_release: code, scenarios and raw rollouts stated as released
  extraction_scope: 'Section 5.3: the spotlighting comparison against the weak inline instruction for Llama-3.2-1B and Qwen2.5-1.5B at Q4 and Q8 (text values); the abstract claim (the quantization, size and style analyses
    test no defence and are not coded)'
defences:
- id: inline_l1q4
  name: one-line inline instruction not to obey commands inside tool results (the harness default)
  origin: this_report
  base_checkpoint: Llama-3.2-1B (Q4)
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: 'single-turn tool-use harness: JSON actions, injection in a tool result'
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: supported
    agent_tool_result: supported
  anchor: includes one line of inline defense
- id: spot_l1q4
  name: hardened spotlighting (tool output wrapped in untrusted-data delimiters with a security preamble)
  origin: this_report
  base_checkpoint: Llama-3.2-1B (Q4)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 11
  layout: 'single-turn tool-use harness: JSON actions, injection in a tool result; tool output delimited'
  layout_compat: *id001
  anchor: wrapping tool output in explicit untrusted-data delimiters
- id: inline_l1q8
  name: one-line inline instruction not to obey commands inside tool results (the harness default)
  origin: this_report
  base_checkpoint: Llama-3.2-1B (Q8)
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: 'single-turn tool-use harness: JSON actions, injection in a tool result'
  layout_compat: *id001
  anchor: includes one line of inline defense
- id: spot_l1q8
  name: hardened spotlighting (tool output wrapped in untrusted-data delimiters with a security preamble)
  origin: this_report
  base_checkpoint: Llama-3.2-1B (Q8)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 11
  layout: 'single-turn tool-use harness: JSON actions, injection in a tool result; tool output delimited'
  layout_compat: *id001
  anchor: wrapping tool output in explicit untrusted-data delimiters
- id: inline_q15q4
  name: one-line inline instruction not to obey commands inside tool results (the harness default)
  origin: this_report
  base_checkpoint: Qwen2.5-1.5B (Q4)
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: 'single-turn tool-use harness: JSON actions, injection in a tool result'
  layout_compat: *id001
  anchor: includes one line of inline defense
- id: spot_q15q4
  name: hardened spotlighting (tool output wrapped in untrusted-data delimiters with a security preamble)
  origin: this_report
  base_checkpoint: Qwen2.5-1.5B (Q4)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 11
  layout: 'single-turn tool-use harness: JSON actions, injection in a tool result; tool output delimited'
  layout_compat: *id001
  anchor: wrapping tool output in explicit untrusted-data delimiters
- id: inline_q15q8
  name: one-line inline instruction not to obey commands inside tool results (the harness default)
  origin: this_report
  base_checkpoint: Qwen2.5-1.5B (Q8)
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: 'single-turn tool-use harness: JSON actions, injection in a tool result'
  layout_compat: *id001
  anchor: includes one line of inline defense
- id: spot_q15q8
  name: hardened spotlighting (tool output wrapped in untrusted-data delimiters with a security preamble)
  origin: this_report
  base_checkpoint: Qwen2.5-1.5B (Q8)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 11
  layout: 'single-turn tool-use harness: JSON actions, injection in a tool result; tool output delimited'
  layout_compat: *id001
  anchor: wrapping tool output in explicit untrusted-data delimiters
evaluations:
- id: ev_l1q4_inline
  defence_id: inline_l1q4
  benchmark: author-built indirect injection suite (eight styles, six tool domains), ten seeds
  attacker_type: static
  attacker_access: black_box
  judge: 'four-outcome scorer: the malicious action was emitted'
  layout_same_as_defence: 'yes'
  measured_by: smallagentsq_zenodo2026
  page: 11
  decoding: temperature 0.7, ten seeds
  pages:
    decoding: 9
- id: ev_l1q4_spot
  defence_id: spot_l1q4
  benchmark: author-built indirect injection suite (eight styles, six tool domains), ten seeds
  attacker_type: static
  attacker_access: black_box
  judge: 'four-outcome scorer: the malicious action was emitted'
  layout_same_as_defence: 'yes'
  measured_by: smallagentsq_zenodo2026
  page: 11
  decoding: temperature 0.7, ten seeds
  pages:
    decoding: 9
- id: ev_l1q8_inline
  defence_id: inline_l1q8
  benchmark: author-built indirect injection suite (eight styles, six tool domains), ten seeds
  attacker_type: static
  attacker_access: black_box
  judge: 'four-outcome scorer: the malicious action was emitted'
  layout_same_as_defence: 'yes'
  measured_by: smallagentsq_zenodo2026
  page: 11
  decoding: temperature 0.7, ten seeds
  pages:
    decoding: 9
- id: ev_l1q8_spot
  defence_id: spot_l1q8
  benchmark: author-built indirect injection suite (eight styles, six tool domains), ten seeds
  attacker_type: static
  attacker_access: black_box
  judge: 'four-outcome scorer: the malicious action was emitted'
  layout_same_as_defence: 'yes'
  measured_by: smallagentsq_zenodo2026
  page: 11
  decoding: temperature 0.7, ten seeds
  pages:
    decoding: 9
- id: ev_q15q4_inline
  defence_id: inline_q15q4
  benchmark: author-built indirect injection suite (eight styles, six tool domains), ten seeds
  attacker_type: static
  attacker_access: black_box
  judge: 'four-outcome scorer: the malicious action was emitted'
  layout_same_as_defence: 'yes'
  measured_by: smallagentsq_zenodo2026
  page: 11
  decoding: temperature 0.7, ten seeds
  pages:
    decoding: 9
- id: ev_q15q4_spot
  defence_id: spot_q15q4
  benchmark: author-built indirect injection suite (eight styles, six tool domains), ten seeds
  attacker_type: static
  attacker_access: black_box
  judge: 'four-outcome scorer: the malicious action was emitted'
  layout_same_as_defence: 'yes'
  measured_by: smallagentsq_zenodo2026
  page: 11
  decoding: temperature 0.7, ten seeds
  pages:
    decoding: 9
- id: ev_q15q8_inline
  defence_id: inline_q15q8
  benchmark: author-built indirect injection suite (eight styles, six tool domains), ten seeds
  attacker_type: static
  attacker_access: black_box
  judge: 'four-outcome scorer: the malicious action was emitted'
  layout_same_as_defence: 'yes'
  measured_by: smallagentsq_zenodo2026
  page: 11
  decoding: temperature 0.7, ten seeds
  pages:
    decoding: 9
- id: ev_q15q8_spot
  defence_id: spot_q15q8
  benchmark: author-built indirect injection suite (eight styles, six tool domains), ten seeds
  attacker_type: static
  attacker_access: black_box
  judge: 'four-outcome scorer: the malicious action was emitted'
  layout_same_as_defence: 'yes'
  measured_by: smallagentsq_zenodo2026
  page: 11
  decoding: temperature 0.7, ten seeds
  pages:
    decoding: 9
results:
- id: r_l1q4_inline
  evaluation_id: ev_l1q4_inline
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '50'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 11
  location: Section 5.3
- id: r_l1q4_spot
  evaluation_id: ev_l1q4_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '29'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 11
  location: Section 5.3
- id: r_l1q8_inline
  evaluation_id: ev_l1q8_inline
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '48'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 11
  location: Section 5.3
- id: r_l1q8_spot
  evaluation_id: ev_l1q8_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '24'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 11
  location: Section 5.3
- id: r_q15q4_inline
  evaluation_id: ev_q15q4_inline
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '42'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 11
  location: Section 5.3
- id: r_q15q4_spot
  evaluation_id: ev_q15q4_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '56'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 11
  location: Section 5.3
- id: r_q15q8_inline
  evaluation_id: ev_q15q8_inline
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '35'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 11
  location: Section 5.3
- id: r_q15q8_spot
  evaluation_id: ev_q15q8_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '53'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 11
  location: Section 5.3
comparisons:
- id: c_l1q4
  left: ev_l1q4_spot
  right: ev_l1q4_inline
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: hardened spotlighting in place of the one-line inline instruction
  page: 11
- id: c_l1q8
  left: ev_l1q8_spot
  right: ev_l1q8_inline
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: hardened spotlighting in place of the one-line inline instruction
  page: 11
- id: c_q15q4
  left: ev_q15q4_spot
  right: ev_q15q4_inline
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: hardened spotlighting in place of the one-line inline instruction
  page: 11
- id: c_q15q8
  left: ev_q15q8_spot
  right: ev_q15q8_inline
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: hardened spotlighting in place of the one-line inline instruction
  page: 11
claims:
- id: k1
  text: spotlighting with delimiters reduces attack success for one architecture while increasing it for another
  location: abstract
  anchor: it reduces attack success for one architecture while increasing it for another
  jobs:
  - R1
  supported_by:
  - c_l1q4
  - c_l1q8
  - c_q15q4
  - c_q15q8
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Section 5.3 and Figure 5
    R2clean: benign-baseline completion is reported only for the default prompt
    R2attack: not reported
schema_gaps:
- where: evaluations ev_*_inline
  note: 'there is no undefended arm: the reference is the harness''s one-line inline instruction, so the comparison is between two prompting defences'

`poisonedtools_zenodo2026` (doi_10_5281_zenodo_21349713)

In [ ]:
%%writefile reports/records/poisonedtools_zenodo2026.yaml
report:
  id: poisonedtools_zenodo2026
  slug: doi_10_5281_zenodo_21349713
  title: 'Poisoned Tools: Prompt-Injection Robustness and Failure Anatomy of Small Open-Weight Tool Agents'
  first_author: Dhruvil Shah
  venue: Zenodo preprint
  peer_reviewed: 'no'
  doi: 10.5281/zenodo.21349713
  version_read: Zenodo version (July 2026)
  first_date: '2026-07-14'
  read: F
  role: re_measures
  code_release: https://github.com/DhruvilShah22/poisoned-tools (p. 1)
  extraction_scope: 'Section 5.4: Table 2 rows D1 (spotlighting), D2 (goal reminder) and D4 (instruction hierarchy), as changes against D0 pooled over five models; the abstract claim (D3 quarantine strips fields and is a sanitiser;
    the composition of D5 is not stated precisely; the attack-surface screen tests no defence)'
defences:
- id: d1
  name: D1 spotlighting (tool results wrapped in typed markers with a standing data instruction)
  origin: other_report
  base_checkpoint: five models pooled (Qwen2.5-Instruct 1.5B, 3B, 7B and Llama-3.2 1B, 3B; 8-bit)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: small ReAct loop over native tool calling; one payload written into one tool-output field per episode
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: supported
    agent_tool_result: supported
  anchor: The defenses are six training-free, inference-time wrappers
- id: d2
  name: D2 goal reminder (the user's request restated after every tool observation)
  origin: other_report
  base_checkpoint: five models pooled (Qwen2.5-Instruct 1.5B, 3B, 7B and Llama-3.2 1B, 3B; 8-bit)
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: small ReAct loop over native tool calling; one payload written into one tool-output field per episode
  layout_compat: *id001
  anchor: The defenses are six training-free, inference-time wrappers
- id: d4
  name: D4 explicit instruction hierarchy in the system prompt
  origin: other_report
  base_checkpoint: five models pooled (Qwen2.5-Instruct 1.5B, 3B, 7B and Llama-3.2 1B, 3B; 8-bit)
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: small ReAct loop over native tool calling; one payload written into one tool-output field per episode
  layout_compat: *id001
  anchor: The defenses are six training-free, inference-time wrappers
evaluations:
- id: ev_d0_att
  defence_id: none:five models pooled (Qwen2.5-Instruct 1.5B, 3B, 7B and Llama-3.2 1B, 3B; 8-bit)
  benchmark: 'focused matrix: unauthorised tool call and data-leak goals, two strongest framings, five placements'
  attacker_type: static
  attacker_access: black_box
  judge: deterministic log-based detection of the planted token or unauthorised call
  layout_same_as_defence: 'yes'
  measured_by: poisonedtools_zenodo2026
  page: 4
  decoding: temperature 0.7, eight seeds (four for 7B)
  baseline_kind: off_the_shelf
- id: ev_d0_ben
  defence_id: none:five models pooled (Qwen2.5-Instruct 1.5B, 3B, 7B and Llama-3.2 1B, 3B; 8-bit)
  benchmark: 'benign control: host tasks without injection'
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic grading of the benign answer
  layout_same_as_defence: 'yes'
  measured_by: poisonedtools_zenodo2026
  page: 4
  decoding: temperature 0.7
  baseline_kind: off_the_shelf
- id: ev_d1_att
  defence_id: d1
  benchmark: 'focused matrix: unauthorised tool call and data-leak goals, two strongest framings, five placements'
  attacker_type: static
  attacker_access: black_box
  judge: deterministic log-based detection of the planted token or unauthorised call
  layout_same_as_defence: 'yes'
  measured_by: poisonedtools_zenodo2026
  page: 9
  decoding: temperature 0.7, eight seeds (four for 7B)
- id: ev_d1_ben
  defence_id: d1
  benchmark: 'benign control: host tasks without injection'
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic grading of the benign answer
  layout_same_as_defence: 'yes'
  measured_by: poisonedtools_zenodo2026
  page: 9
  decoding: temperature 0.7
- id: ev_d2_att
  defence_id: d2
  benchmark: 'focused matrix: unauthorised tool call and data-leak goals, two strongest framings, five placements'
  attacker_type: static
  attacker_access: black_box
  judge: deterministic log-based detection of the planted token or unauthorised call
  layout_same_as_defence: 'yes'
  measured_by: poisonedtools_zenodo2026
  page: 9
  decoding: temperature 0.7, eight seeds (four for 7B)
- id: ev_d2_ben
  defence_id: d2
  benchmark: 'benign control: host tasks without injection'
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic grading of the benign answer
  layout_same_as_defence: 'yes'
  measured_by: poisonedtools_zenodo2026
  page: 9
  decoding: temperature 0.7
- id: ev_d4_att
  defence_id: d4
  benchmark: 'focused matrix: unauthorised tool call and data-leak goals, two strongest framings, five placements'
  attacker_type: static
  attacker_access: black_box
  judge: deterministic log-based detection of the planted token or unauthorised call
  layout_same_as_defence: 'yes'
  measured_by: poisonedtools_zenodo2026
  page: 9
  decoding: temperature 0.7, eight seeds (four for 7B)
- id: ev_d4_ben
  defence_id: d4
  benchmark: 'benign control: host tasks without injection'
  attacker_type: none
  attacker_access: not_applicable
  judge: deterministic grading of the benign answer
  layout_same_as_defence: 'yes'
  measured_by: poisonedtools_zenodo2026
  page: 9
  decoding: temperature 0.7
results:
- id: r_d1_att
  evaluation_id: ev_d1_att
  quantity: R1
  coverage: direct
  metric_as_printed: change in attack success against D0 (pooled, given exposure)
  value: '+0.024'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 2
- id: r_d1_ben
  evaluation_id: ev_d1_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: change in benign task success against D0
  value: '-0.051'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 2
- id: r_d2_att
  evaluation_id: ev_d2_att
  quantity: R1
  coverage: direct
  metric_as_printed: change in attack success against D0 (pooled, given exposure)
  value: '-0.056'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 2
- id: r_d2_ben
  evaluation_id: ev_d2_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: change in benign task success against D0
  value: '-0.194'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 2
- id: r_d4_att
  evaluation_id: ev_d4_att
  quantity: R1
  coverage: direct
  metric_as_printed: change in attack success against D0 (pooled, given exposure)
  value: '-0.038'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 2
- id: r_d4_ben
  evaluation_id: ev_d4_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: change in benign task success against D0
  value: '-0.028'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 2
comparisons:
- id: c_d1_att
  left: ev_d1_att
  right: ev_d0_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 9
- id: c_d1_ben
  left: ev_d1_ben
  right: ev_d0_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 9
- id: c_d2_att
  left: ev_d2_att
  right: ev_d0_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 9
- id: c_d2_ben
  left: ev_d2_ben
  right: ev_d0_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 9
- id: c_d4_att
  left: ev_d4_att
  right: ev_d0_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 9
- id: c_d4_ben
  left: ev_d4_ben
  right: ev_d0_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 9
claims:
- id: k1
  text: none of the cheap training-free defences significantly reduces attack success, and the goal reminder costs nineteen points of benign utility
  location: abstract
  anchor: none of the cheap training-free defenses we tested significantly reduces
  jobs:
  - R1
  - R2
  supported_by:
  - c_d1_att
  - c_d2_att
  - c_d4_att
  - c_d2_ben
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 2 (changes against D0, pooled)
    R2clean: Table 2 (changes in benign success)
    R2attack: not reported
schema_gaps:
- where: results r_*
  note: Table 2 gives only paired changes against the undefended baseline, pooled over five models; the defended results carry those changes and the undefended evaluations carry no value; Holm-corrected p-values are 0.10 or
    above for every attack-success change

`msacore_zenodo2026` (doi_10_5281_zenodo_23506227)

In [ ]:
%%writefile reports/records/msacore_zenodo2026.yaml
report:
  id: msacore_zenodo2026
  slug: doi_10_5281_zenodo_23506227
  title: 'The MLPs of Layers 8-15 of Meta-SecAlign-8B and Its Scaling Knob: An Unresolved Registered Test'
  first_author: Kazuki Nakayashiki
  venue: Zenodo preprint
  peer_reviewed: 'no'
  doi: 10.5281/zenodo.23506227
  version_read: Zenodo version
  first_date: '2026-10-20'
  read: F
  role: re_measures
  code_release: frozen packages with manifests (Table 7)
  extraction_scope: 'Section 4 (second, registered study on 3,000 fresh SEP items): undefended, full adapter, CORE only, adapter without CORE and the window four layers earlier; the abstract claim (the knob curve of Table
    3, the first study''s mask screen of Table 6 and the trusted-memory contrast, which concerns memory following rather than injection, not coded)'
defences:
- id: msa8
  name: Meta-SecAlign-8B (released LoRA adapter, full knob)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  layout: Meta-SecAlign input role for SEP data
  layout_compat: &id001
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: untested
  anchor: against 2.9% with the whole adapter and 72.5% without it
- id: core
  name: Meta-SecAlign-8B restricted to CORE (gate, up and down projections of layers 8-15; 20.1% of the adapter's parameters)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  layout: Meta-SecAlign input role for SEP data
  layout_compat: *id001
  anchor: We call CORE the gate, up and down projections of layers 8–15
- id: nocore
  name: Meta-SecAlign-8B with CORE removed
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  layout: Meta-SecAlign input role for SEP data
  layout_compat: *id001
  anchor: We call CORE the gate, up and down projections of layers 8–15
- id: shift
  name: Meta-SecAlign-8B restricted to the same MLP projections of layers 4-11
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  layout: Meta-SecAlign input role for SEP data
  layout_compat: *id001
  anchor: We call CORE the gate, up and down projections of layers 8–15
evaluations:
- id: ev_none
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: SEP, 3,000 items not served under attack in any earlier run (non-adaptive SEP attacks)
  attacker_type: static
  attacker_access: black_box
  judge: SEP witness judged on responses up to 512 tokens
  layout_same_as_defence: 'yes'
  measured_by: msacore_zenodo2026
  page: 6
  decoding: 512 response tokens; sampling not stated
  sample_size: 3,000
  baseline_kind: off_the_shelf
- id: ev_msa8
  defence_id: msa8
  benchmark: SEP, 3,000 items not served under attack in any earlier run (non-adaptive SEP attacks)
  attacker_type: static
  attacker_access: black_box
  judge: SEP witness judged on responses up to 512 tokens
  layout_same_as_defence: 'yes'
  measured_by: msacore_zenodo2026
  page: 6
  decoding: 512 response tokens; sampling not stated
  sample_size: 3,000
- id: ev_core
  defence_id: core
  benchmark: SEP, 3,000 items not served under attack in any earlier run (non-adaptive SEP attacks)
  attacker_type: static
  attacker_access: black_box
  judge: SEP witness judged on responses up to 512 tokens
  layout_same_as_defence: 'yes'
  measured_by: msacore_zenodo2026
  page: 6
  decoding: 512 response tokens; sampling not stated
  sample_size: 3,000
- id: ev_nocore
  defence_id: nocore
  benchmark: SEP, 3,000 items not served under attack in any earlier run (non-adaptive SEP attacks)
  attacker_type: static
  attacker_access: black_box
  judge: SEP witness judged on responses up to 512 tokens
  layout_same_as_defence: 'yes'
  measured_by: msacore_zenodo2026
  page: 6
  decoding: 512 response tokens; sampling not stated
  sample_size: 3,000
- id: ev_shift
  defence_id: shift
  benchmark: SEP, 3,000 items not served under attack in any earlier run (non-adaptive SEP attacks)
  attacker_type: static
  attacker_access: black_box
  judge: SEP witness judged on responses up to 512 tokens
  layout_same_as_defence: 'yes'
  measured_by: msacore_zenodo2026
  page: 6
  decoding: 512 response tokens; sampling not stated
  sample_size: 3,000
results:
- id: r_none
  evaluation_id: ev_none
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '72.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 6
  location: Section 4
- id: r_msa8
  evaluation_id: ev_msa8
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '2.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 6
  location: Section 4
- id: r_core
  evaluation_id: ev_core
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '16.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 6
  location: Section 4
- id: r_nocore
  evaluation_id: ev_nocore
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '27.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 6
  location: Section 4
- id: r_shift
  evaluation_id: ev_shift
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '10.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 6
  location: Section 4
- id: r_core_task
  evaluation_id: ev_core
  quantity: R2attack
  coverage: proxy
  metric_as_printed: main-task proxy held under attack
  value: '73.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 6
  location: Section 4
- id: r_msa8_task
  evaluation_id: ev_msa8
  quantity: R2attack
  coverage: proxy
  metric_as_printed: main-task proxy held under attack
  value: '73.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 6
  location: Section 4
comparisons:
- id: c_msa8
  left: ev_msa8
  right: ev_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_core_none
  left: ev_core
  right: ev_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_core_msa8
  left: ev_core
  right: ev_msa8
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the adapter with only CORE kept, against the whole adapter
  page: 6
- id: c_nocore_msa8
  left: ev_nocore
  right: ev_msa8
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the adapter with CORE removed, against the whole adapter
  page: 6
- id: c_shift_msa8
  left: ev_shift
  right: ev_msa8
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the adapter with only the layers 4-11 projections kept, against the whole adapter
  page: 6
claims:
- id: k1
  text: keeping only the MLPs of layers 8-15 kept 80.7% of the adapter's protection (16.3% attack success against 2.9% with the whole adapter)
  location: abstract
  anchor: Keeping only the part kept 80.7%
  jobs:
  - R1
  supported_by:
  - c_core_msa8
  - c_core_none
  - c_msa8
  page: 1
  contradicted_by:
  - r_shift
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: proxy
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Section 4
    R2attack: main-task proxy under attack (Section 4)
    R2clean: not reported
schema_gaps:
- where: report
  note: 'a component study of one released adapter: the restricted adapters are cut from Meta-SecAlign-8B, not retrained; the paper''s side effect of interest is a trusted-memory following contrast, which the schema has no
    field for'

`whosewords_zenodo2026` (doi_10_5281_zenodo_23758884)

In [ ]:
%%writefile reports/records/whosewords_zenodo2026.yaml
report:
  id: whosewords_zenodo2026
  slug: doi_10_5281_zenodo_23758884
  title: Whose words count? Chat role and partial control of goal adherence
  first_author: Sam Ramdan
  venue: Zenodo preprint
  peer_reviewed: 'no'
  doi: 10.5281/zenodo.23758884
  version_read: Zenodo v2 (30 October 2026)
  first_date: '2026-10-29'
  read: F
  role: mixed
  code_release: https://github.com/Somebodyhere101/whose-words-count
  extraction_scope: 'Section 8: the role-direction steering test against an injected instruction in a document (Qwen3-4B, 4-bit), with its random-direction control; (the goal-adherence experiments, which concern trusted goal
    reminders rather than injected data, and the fine-tuning pilots are not coded)'
defences:
- id: rolesteer
  name: role-direction steering (assistant-minus-system/user activation direction added to the injected instruction)
  origin: this_report
  base_checkpoint: Qwen3-4B (4-bit)
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: summarisation of a document that carries an injected instruction; the steering vector is added to the injected instruction's tokens
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: The direction is added to an injected instruction inside a document
- id: randdir
  name: a random direction of the same norm (control)
  origin: this_report
  base_checkpoint: Qwen3-4B (4-bit)
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 16
  layout: summarisation of a document that carries an injected instruction; the steering vector is added to the injected instruction's tokens
  anchor: The single random control gave 0.642
evaluations:
- id: ev_none
  defence_id: none:Qwen3-4B (4-bit)
  benchmark: author-built summarisation task with an instruction injected into the document (INJ2_PREREG)
  attacker_type: static
  attacker_access: black_box
  judge: compliance with the injected instruction (rule logged before data)
  layout_same_as_defence: 'yes'
  measured_by: whosewords_zenodo2026
  page: 16
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_steer
  defence_id: rolesteer
  benchmark: author-built summarisation task with an instruction injected into the document (INJ2_PREREG)
  attacker_type: static
  attacker_access: black_box
  judge: compliance with the injected instruction (rule logged before data)
  layout_same_as_defence: 'yes'
  measured_by: whosewords_zenodo2026
  page: 16
  decoding: not reported
- id: ev_rand
  defence_id: randdir
  benchmark: author-built summarisation task with an instruction injected into the document (INJ2_PREREG)
  attacker_type: static
  attacker_access: black_box
  judge: compliance with the injected instruction (rule logged before data)
  layout_same_as_defence: 'yes'
  measured_by: whosewords_zenodo2026
  page: 16
  decoding: not reported
results:
- id: r_none
  evaluation_id: ev_none
  quantity: R1
  coverage: direct
  metric_as_printed: compliance with the injected instruction
  value: '0.209'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 16
  location: Section 8
- id: r_steer
  evaluation_id: ev_steer
  quantity: R1
  coverage: direct
  metric_as_printed: compliance with the injected instruction
  value: '0.000'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 16
  location: Section 8
- id: r_rand
  evaluation_id: ev_rand
  quantity: R1
  coverage: direct
  metric_as_printed: compliance with the injected instruction
  value: '0.642'
  unit: fraction
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 16
  location: Section 8
- id: r_none_legit
  evaluation_id: ev_none
  quantity: R2attack
  coverage: proxy
  metric_as_printed: legitimate instruction followed
  value: '0.79'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 16
  location: Section 8
- id: r_steer_legit
  evaluation_id: ev_steer
  quantity: R2attack
  coverage: proxy
  metric_as_printed: legitimate instruction followed
  value: '1.00'
  unit: fraction
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 16
  location: Section 8
comparisons:
- id: c_steer
  left: ev_steer
  right: ev_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_steer_rand
  left: ev_steer
  right: ev_rand
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the role direction against a random direction of the same norm
  page: 16
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: proxy
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Section 8
    R2attack: legitimate instruction followed (Section 8)
    R2clean: not reported
schema_gaps:
- where: report
  note: the injection test is one model in 4-bit (in bf16 the model ignores the injection at baseline) and is presented by the author as a replication of prior steering work, not a contribution; the exploratory random-direction
    check places the single control draw at the 91st percentile; the study is described as AI-run

`whosaidthis_zenodo2026` (doi_10_5281_zenodo_23758886)

In [ ]:
%%writefile reports/records/whosaidthis_zenodo2026.yaml
report:
  id: whosaidthis_zenodo2026
  slug: doi_10_5281_zenodo_23758886
  title: Who said this? Most chat models weigh a text's source differently for each behaviour, and a steering lever against injected instructions
  first_author: Sam Ramdan
  venue: Zenodo preprint
  peer_reviewed: 'no'
  doi: 10.5281/zenodo.23758886
  version_read: Zenodo v2 (30 October 2026)
  first_date: '2026-10-29'
  read: F
  role: proposes
  code_release: https://github.com/Somebodyhere101/whose-words-count
  extraction_scope: 'Sections 3.7 and 3.8: the steering lever against no defence on SEP and InjecAgent for twelve models (the two result tables); the abstract claim (the source-ranking and patching analyses test no defence;
    the prompt-defence and random-direction comparisons are described in the text without per-model numbers and are not coded)'
defences:
- id: lever_q8
  name: 'steering lever: amplified user-minus-tool direction added to tool-output tokens at about 0.45 depth (found by search on synthetic items)'
  origin: this_report
  base_checkpoint: Qwen3-8B
  axis1:
  - chat_roles
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 12
  layout: task in the system turn; data in a 'Tool output:' user turn (SEP); InjecAgent's own ReAct prompt in one user turn
  layout_compat: &id001
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: 'Result: LEVER_FOUND (the winner passes in 3 of 4 models)'
- id: lever_q17
  name: 'steering lever: amplified user-minus-tool direction added to tool-output tokens at about 0.45 depth (found by search on synthetic items)'
  origin: this_report
  base_checkpoint: Qwen3-1.7B
  axis1:
  - chat_roles
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 12
  layout: task in the system turn; data in a 'Tool output:' user turn (SEP); InjecAgent's own ReAct prompt in one user turn
  layout_compat: *id001
  anchor: 'Result: LEVER_FOUND (the winner passes in 3 of 4 models)'
- id: lever_smol
  name: 'steering lever: amplified user-minus-tool direction added to tool-output tokens at about 0.45 depth (found by search on synthetic items)'
  origin: this_report
  base_checkpoint: SmolLM3-3B
  axis1:
  - chat_roles
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 12
  layout: task in the system turn; data in a 'Tool output:' user turn (SEP); InjecAgent's own ReAct prompt in one user turn
  layout_compat: *id001
  anchor: 'Result: LEVER_FOUND (the winner passes in 3 of 4 models)'
- id: lever_gran
  name: 'steering lever: amplified user-minus-tool direction added to tool-output tokens at about 0.45 depth (found by search on synthetic items)'
  origin: this_report
  base_checkpoint: Granite-3.3-8B
  axis1:
  - chat_roles
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 12
  layout: task in the system turn; data in a 'Tool output:' user turn (SEP); InjecAgent's own ReAct prompt in one user turn
  layout_compat: *id001
  anchor: 'Result: LEVER_FOUND (the winner passes in 3 of 4 models)'
- id: lever_olmo7
  name: 'steering lever: amplified user-minus-tool direction added to tool-output tokens at about 0.45 depth (found by search on synthetic items)'
  origin: this_report
  base_checkpoint: OLMo-2-7B
  axis1:
  - chat_roles
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 12
  layout: task in the system turn; data in a 'Tool output:' user turn (SEP); InjecAgent's own ReAct prompt in one user turn
  layout_compat: *id001
  anchor: 'Result: LEVER_FOUND (the winner passes in 3 of 4 models)'
- id: lever_phi4
  name: 'steering lever: amplified user-minus-tool direction added to tool-output tokens at about 0.45 depth (found by search on synthetic items)'
  origin: this_report
  base_checkpoint: Phi-4
  axis1:
  - chat_roles
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 12
  layout: task in the system turn; data in a 'Tool output:' user turn (SEP); InjecAgent's own ReAct prompt in one user turn
  layout_compat: *id001
  anchor: 'Result: LEVER_FOUND (the winner passes in 3 of 4 models)'
- id: lever_q14
  name: 'steering lever: amplified user-minus-tool direction added to tool-output tokens at about 0.45 depth (found by search on synthetic items)'
  origin: this_report
  base_checkpoint: Qwen3-14B
  axis1:
  - chat_roles
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 12
  layout: task in the system turn; data in a 'Tool output:' user turn (SEP); InjecAgent's own ReAct prompt in one user turn
  layout_compat: *id001
  anchor: 'Result: LEVER_FOUND (the winner passes in 3 of 4 models)'
- id: lever_q32
  name: 'steering lever: amplified user-minus-tool direction added to tool-output tokens at about 0.45 depth (found by search on synthetic items)'
  origin: this_report
  base_checkpoint: Qwen3-32B
  axis1:
  - chat_roles
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 12
  layout: task in the system turn; data in a 'Tool output:' user turn (SEP); InjecAgent's own ReAct prompt in one user turn
  layout_compat: *id001
  anchor: 'Result: LEVER_FOUND (the winner passes in 3 of 4 models)'
- id: lever_fal
  name: 'steering lever: amplified user-minus-tool direction added to tool-output tokens at about 0.45 depth (found by search on synthetic items)'
  origin: this_report
  base_checkpoint: Falcon3-10B
  axis1:
  - chat_roles
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 12
  layout: task in the system turn; data in a 'Tool output:' user turn (SEP); InjecAgent's own ReAct prompt in one user turn
  layout_compat: *id001
  anchor: 'Result: LEVER_FOUND (the winner passes in 3 of 4 models)'
- id: lever_yi
  name: 'steering lever: amplified user-minus-tool direction added to tool-output tokens at about 0.45 depth (found by search on synthetic items)'
  origin: this_report
  base_checkpoint: Yi-1.5-9B
  axis1:
  - chat_roles
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 12
  layout: task in the system turn; data in a 'Tool output:' user turn (SEP); InjecAgent's own ReAct prompt in one user turn
  layout_compat: *id001
  anchor: 'Result: LEVER_FOUND (the winner passes in 3 of 4 models)'
- id: lever_olmo13
  name: 'steering lever: amplified user-minus-tool direction added to tool-output tokens at about 0.45 depth (found by search on synthetic items)'
  origin: this_report
  base_checkpoint: OLMo-2-13B
  axis1:
  - chat_roles
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 12
  layout: task in the system turn; data in a 'Tool output:' user turn (SEP); InjecAgent's own ReAct prompt in one user turn
  layout_compat: *id001
  anchor: 'Result: LEVER_FOUND (the winner passes in 3 of 4 models)'
- id: lever_olmo32
  name: 'steering lever: amplified user-minus-tool direction added to tool-output tokens at about 0.45 depth (found by search on synthetic items)'
  origin: this_report
  base_checkpoint: OLMo-2-32B
  axis1:
  - chat_roles
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 12
  layout: task in the system turn; data in a 'Tool output:' user turn (SEP); InjecAgent's own ReAct prompt in one user turn
  layout_compat: *id001
  anchor: 'Result: LEVER_FOUND (the winner passes in 3 of 4 models)'
evaluations:
- id: ev_q8_none_sep
  defence_id: none:Qwen3-8B
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '300'
  baseline_kind: off_the_shelf
- id: ev_q8_lever_sep
  defence_id: lever_q8
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '300'
- id: ev_q8_none_ia
  defence_id: none:Qwen3-8B
  benchmark: InjecAgent, 300 cases, repository ReAct prompt
  attacker_type: static
  attacker_access: black_box
  judge: repository rules, valid replies only
  layout_same_as_defence: 'no'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: 300 (valid replies vary by condition)
  baseline_kind: off_the_shelf
- id: ev_q8_lever_ia
  defence_id: lever_q8
  benchmark: InjecAgent, 300 cases, repository ReAct prompt
  attacker_type: static
  attacker_access: black_box
  judge: repository rules, valid replies only
  layout_same_as_defence: 'no'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: 300 (valid replies vary by condition)
- id: ev_q17_none_sep
  defence_id: none:Qwen3-1.7B
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '300'
  baseline_kind: off_the_shelf
- id: ev_q17_lever_sep
  defence_id: lever_q17
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '300'
- id: ev_q17_none_ia
  defence_id: none:Qwen3-1.7B
  benchmark: InjecAgent, 300 cases, repository ReAct prompt
  attacker_type: static
  attacker_access: black_box
  judge: repository rules, valid replies only
  layout_same_as_defence: 'no'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: 300 (valid replies vary by condition)
  baseline_kind: off_the_shelf
- id: ev_q17_lever_ia
  defence_id: lever_q17
  benchmark: InjecAgent, 300 cases, repository ReAct prompt
  attacker_type: static
  attacker_access: black_box
  judge: repository rules, valid replies only
  layout_same_as_defence: 'no'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: 300 (valid replies vary by condition)
- id: ev_smol_none_sep
  defence_id: none:SmolLM3-3B
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '300'
  baseline_kind: off_the_shelf
- id: ev_smol_lever_sep
  defence_id: lever_smol
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '300'
- id: ev_smol_none_ia
  defence_id: none:SmolLM3-3B
  benchmark: InjecAgent, 300 cases, repository ReAct prompt
  attacker_type: static
  attacker_access: black_box
  judge: repository rules, valid replies only
  layout_same_as_defence: 'no'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: 300 (valid replies vary by condition)
  baseline_kind: off_the_shelf
- id: ev_smol_lever_ia
  defence_id: lever_smol
  benchmark: InjecAgent, 300 cases, repository ReAct prompt
  attacker_type: static
  attacker_access: black_box
  judge: repository rules, valid replies only
  layout_same_as_defence: 'no'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: 300 (valid replies vary by condition)
- id: ev_gran_none_sep
  defence_id: none:Granite-3.3-8B
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '300'
  baseline_kind: off_the_shelf
- id: ev_gran_lever_sep
  defence_id: lever_gran
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '300'
- id: ev_gran_none_ia
  defence_id: none:Granite-3.3-8B
  benchmark: InjecAgent, 300 cases, repository ReAct prompt
  attacker_type: static
  attacker_access: black_box
  judge: repository rules, valid replies only
  layout_same_as_defence: 'no'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: 300 (valid replies vary by condition)
  baseline_kind: off_the_shelf
- id: ev_gran_lever_ia
  defence_id: lever_gran
  benchmark: InjecAgent, 300 cases, repository ReAct prompt
  attacker_type: static
  attacker_access: black_box
  judge: repository rules, valid replies only
  layout_same_as_defence: 'no'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: 300 (valid replies vary by condition)
- id: ev_olmo7_none_sep
  defence_id: none:OLMo-2-7B
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '300'
  baseline_kind: off_the_shelf
- id: ev_olmo7_lever_sep
  defence_id: lever_olmo7
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '300'
- id: ev_olmo7_none_ia
  defence_id: none:OLMo-2-7B
  benchmark: InjecAgent, 300 cases, repository ReAct prompt
  attacker_type: static
  attacker_access: black_box
  judge: repository rules, valid replies only
  layout_same_as_defence: 'no'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: 300 (valid replies vary by condition)
  baseline_kind: off_the_shelf
- id: ev_olmo7_lever_ia
  defence_id: lever_olmo7
  benchmark: InjecAgent, 300 cases, repository ReAct prompt
  attacker_type: static
  attacker_access: black_box
  judge: repository rules, valid replies only
  layout_same_as_defence: 'no'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: 300 (valid replies vary by condition)
- id: ev_phi4_none_sep
  defence_id: none:Phi-4
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '300'
  baseline_kind: off_the_shelf
- id: ev_phi4_lever_sep
  defence_id: lever_phi4
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: '300'
- id: ev_phi4_none_ia
  defence_id: none:Phi-4
  benchmark: InjecAgent, 300 cases, repository ReAct prompt
  attacker_type: static
  attacker_access: black_box
  judge: repository rules, valid replies only
  layout_same_as_defence: 'no'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: 300 (valid replies vary by condition)
  baseline_kind: off_the_shelf
- id: ev_phi4_lever_ia
  defence_id: lever_phi4
  benchmark: InjecAgent, 300 cases, repository ReAct prompt
  attacker_type: static
  attacker_access: black_box
  judge: repository rules, valid replies only
  layout_same_as_defence: 'no'
  measured_by: whosaidthis_zenodo2026
  page: 14
  decoding: not reported
  sample_size: 300 (valid replies vary by condition)
- id: ev_q14_none_sep
  defence_id: none:Qwen3-14B
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 16
  decoding: not reported
  sample_size: '300'
  baseline_kind: off_the_shelf
- id: ev_q14_lever_sep
  defence_id: lever_q14
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 16
  decoding: not reported
  sample_size: '300'
- id: ev_q14_none_ia
  defence_id: none:Qwen3-14B
  benchmark: InjecAgent, 300 cases, repository ReAct prompt
  attacker_type: static
  attacker_access: black_box
  judge: repository rules, valid replies only
  layout_same_as_defence: 'no'
  measured_by: whosaidthis_zenodo2026
  page: 16
  decoding: not reported
  sample_size: 300 (valid replies vary by condition)
  baseline_kind: off_the_shelf
- id: ev_q14_lever_ia
  defence_id: lever_q14
  benchmark: InjecAgent, 300 cases, repository ReAct prompt
  attacker_type: static
  attacker_access: black_box
  judge: repository rules, valid replies only
  layout_same_as_defence: 'no'
  measured_by: whosaidthis_zenodo2026
  page: 16
  decoding: not reported
  sample_size: 300 (valid replies vary by condition)
- id: ev_q32_none_sep
  defence_id: none:Qwen3-32B
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 16
  decoding: not reported
  sample_size: '300'
  baseline_kind: off_the_shelf
- id: ev_q32_lever_sep
  defence_id: lever_q32
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 16
  decoding: not reported
  sample_size: '300'
- id: ev_q32_none_ia
  defence_id: none:Qwen3-32B
  benchmark: InjecAgent, 300 cases, repository ReAct prompt
  attacker_type: static
  attacker_access: black_box
  judge: repository rules, valid replies only
  layout_same_as_defence: 'no'
  measured_by: whosaidthis_zenodo2026
  page: 16
  decoding: not reported
  sample_size: 300 (valid replies vary by condition)
  baseline_kind: off_the_shelf
- id: ev_q32_lever_ia
  defence_id: lever_q32
  benchmark: InjecAgent, 300 cases, repository ReAct prompt
  attacker_type: static
  attacker_access: black_box
  judge: repository rules, valid replies only
  layout_same_as_defence: 'no'
  measured_by: whosaidthis_zenodo2026
  page: 16
  decoding: not reported
  sample_size: 300 (valid replies vary by condition)
- id: ev_fal_none_sep
  defence_id: none:Falcon3-10B
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 16
  decoding: not reported
  sample_size: '300'
  baseline_kind: off_the_shelf
- id: ev_fal_lever_sep
  defence_id: lever_fal
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 16
  decoding: not reported
  sample_size: '300'
- id: ev_yi_none_sep
  defence_id: none:Yi-1.5-9B
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 16
  decoding: not reported
  sample_size: '300'
  baseline_kind: off_the_shelf
- id: ev_yi_lever_sep
  defence_id: lever_yi
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 16
  decoding: not reported
  sample_size: '300'
- id: ev_olmo13_none_sep
  defence_id: none:OLMo-2-13B
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 16
  decoding: not reported
  sample_size: '300'
  baseline_kind: off_the_shelf
- id: ev_olmo13_lever_sep
  defence_id: lever_olmo13
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 16
  decoding: not reported
  sample_size: '300'
- id: ev_olmo32_none_sep
  defence_id: none:OLMo-2-32B
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 16
  decoding: not reported
  sample_size: '300'
  baseline_kind: off_the_shelf
- id: ev_olmo32_lever_sep
  defence_id: lever_olmo32
  benchmark: SEP, 300 items, data in a 'Tool output:' user turn
  attacker_type: static
  attacker_access: black_box
  judge: probe executed (attack) or task probe executed (utility)
  layout_same_as_defence: 'yes'
  measured_by: whosaidthis_zenodo2026
  page: 16
  decoding: not reported
  sample_size: '300'
results:
- id: r_q8_none_sepasr
  evaluation_id: ev_q8_none_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '30.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_q8_none_seputil
  evaluation_id: ev_q8_none_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '64.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_q8_lever_sepasr
  evaluation_id: ev_q8_lever_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '13.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_q8_lever_seputil
  evaluation_id: ev_q8_lever_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '67.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_q8_none_ia
  evaluation_id: ev_q8_none_ia
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent attack success (valid replies)
  value: '39.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_q8_lever_ia
  evaluation_id: ev_q8_lever_ia
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent attack success (valid replies)
  value: '11.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_q17_none_sepasr
  evaluation_id: ev_q17_none_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '28.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_q17_none_seputil
  evaluation_id: ev_q17_none_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '45.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_q17_lever_sepasr
  evaluation_id: ev_q17_lever_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '5.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_q17_lever_seputil
  evaluation_id: ev_q17_lever_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '47.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_q17_none_ia
  evaluation_id: ev_q17_none_ia
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent attack success (valid replies)
  value: '56.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_q17_lever_ia
  evaluation_id: ev_q17_lever_ia
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent attack success (valid replies)
  value: '3.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_smol_none_sepasr
  evaluation_id: ev_smol_none_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '41.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_smol_none_seputil
  evaluation_id: ev_smol_none_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '48.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_smol_lever_sepasr
  evaluation_id: ev_smol_lever_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '24.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_smol_lever_seputil
  evaluation_id: ev_smol_lever_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '54.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_smol_none_ia
  evaluation_id: ev_smol_none_ia
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent attack success (valid replies)
  value: '3.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_smol_lever_ia
  evaluation_id: ev_smol_lever_ia
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent attack success (valid replies)
  value: '0.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_gran_none_sepasr
  evaluation_id: ev_gran_none_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '44.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_gran_none_seputil
  evaluation_id: ev_gran_none_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '66.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_gran_lever_sepasr
  evaluation_id: ev_gran_lever_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '35.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_gran_lever_seputil
  evaluation_id: ev_gran_lever_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '68.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_gran_none_ia
  evaluation_id: ev_gran_none_ia
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent attack success (valid replies)
  value: '46.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_gran_lever_ia
  evaluation_id: ev_gran_lever_ia
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent attack success (valid replies)
  value: '40.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_olmo7_none_sepasr
  evaluation_id: ev_olmo7_none_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '56.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_olmo7_none_seputil
  evaluation_id: ev_olmo7_none_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '61.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_olmo7_lever_sepasr
  evaluation_id: ev_olmo7_lever_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '44.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_olmo7_lever_seputil
  evaluation_id: ev_olmo7_lever_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '73.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_olmo7_none_ia
  evaluation_id: ev_olmo7_none_ia
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent attack success (valid replies)
  value: '89.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_olmo7_lever_ia
  evaluation_id: ev_olmo7_lever_ia
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent attack success (valid replies)
  value: '87.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_phi4_none_sepasr
  evaluation_id: ev_phi4_none_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '24.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_phi4_none_seputil
  evaluation_id: ev_phi4_none_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '63.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_phi4_lever_sepasr
  evaluation_id: ev_phi4_lever_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '22.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_phi4_lever_seputil
  evaluation_id: ev_phi4_lever_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '65.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_phi4_none_ia
  evaluation_id: ev_phi4_none_ia
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent attack success (valid replies)
  value: '21.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_phi4_lever_ia
  evaluation_id: ev_phi4_lever_ia
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent attack success (valid replies)
  value: '11.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 3.7 or 3.8 table
- id: r_q14_none_sepasr
  evaluation_id: ev_q14_none_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '37.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_q14_none_seputil
  evaluation_id: ev_q14_none_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '71.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_q14_lever_sepasr
  evaluation_id: ev_q14_lever_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '18.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_q14_lever_seputil
  evaluation_id: ev_q14_lever_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '72.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_q14_none_ia
  evaluation_id: ev_q14_none_ia
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent attack success (valid replies)
  value: '29.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_q14_lever_ia
  evaluation_id: ev_q14_lever_ia
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent attack success (valid replies)
  value: '4.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_q32_none_sepasr
  evaluation_id: ev_q32_none_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '40.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_q32_none_seputil
  evaluation_id: ev_q32_none_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '73.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_q32_lever_sepasr
  evaluation_id: ev_q32_lever_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '30.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_q32_lever_seputil
  evaluation_id: ev_q32_lever_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '72.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_q32_none_ia
  evaluation_id: ev_q32_none_ia
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent attack success (valid replies)
  value: '27.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_q32_lever_ia
  evaluation_id: ev_q32_lever_ia
  quantity: R1
  coverage: direct
  metric_as_printed: InjecAgent attack success (valid replies)
  value: '8.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_fal_none_sepasr
  evaluation_id: ev_fal_none_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '48.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_fal_none_seputil
  evaluation_id: ev_fal_none_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '67.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_fal_lever_sepasr
  evaluation_id: ev_fal_lever_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '33.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_fal_lever_seputil
  evaluation_id: ev_fal_lever_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '68.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_yi_none_sepasr
  evaluation_id: ev_yi_none_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '47.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_yi_none_seputil
  evaluation_id: ev_yi_none_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '66.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_yi_lever_sepasr
  evaluation_id: ev_yi_lever_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '30.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_yi_lever_seputil
  evaluation_id: ev_yi_lever_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '66.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_olmo13_none_sepasr
  evaluation_id: ev_olmo13_none_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '43.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_olmo13_none_seputil
  evaluation_id: ev_olmo13_none_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '69.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_olmo13_lever_sepasr
  evaluation_id: ev_olmo13_lever_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '38.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_olmo13_lever_seputil
  evaluation_id: ev_olmo13_lever_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '70.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_olmo32_none_sepasr
  evaluation_id: ev_olmo32_none_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '58.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_olmo32_none_seputil
  evaluation_id: ev_olmo32_none_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '88.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_olmo32_lever_sepasr
  evaluation_id: ev_olmo32_lever_sep
  quantity: R1
  coverage: direct
  metric_as_printed: SEP attack success
  value: '47.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
- id: r_olmo32_lever_seputil
  evaluation_id: ev_olmo32_lever_sep
  quantity: R2attack
  coverage: proxy
  metric_as_printed: SEP utility (task probe executed)
  value: '89.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 16
  location: Section 3.7 or 3.8 table
comparisons:
- id: c_q8_sep
  left: ev_q8_lever_sep
  right: ev_q8_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_q8_ia
  left: ev_q8_lever_ia
  right: ev_q8_none_ia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_q17_sep
  left: ev_q17_lever_sep
  right: ev_q17_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_q17_ia
  left: ev_q17_lever_ia
  right: ev_q17_none_ia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_smol_sep
  left: ev_smol_lever_sep
  right: ev_smol_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_smol_ia
  left: ev_smol_lever_ia
  right: ev_smol_none_ia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_gran_sep
  left: ev_gran_lever_sep
  right: ev_gran_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_gran_ia
  left: ev_gran_lever_ia
  right: ev_gran_none_ia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_olmo7_sep
  left: ev_olmo7_lever_sep
  right: ev_olmo7_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_olmo7_ia
  left: ev_olmo7_lever_ia
  right: ev_olmo7_none_ia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_phi4_sep
  left: ev_phi4_lever_sep
  right: ev_phi4_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_phi4_ia
  left: ev_phi4_lever_ia
  right: ev_phi4_none_ia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
- id: c_q14_sep
  left: ev_q14_lever_sep
  right: ev_q14_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_q14_ia
  left: ev_q14_lever_ia
  right: ev_q14_none_ia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_q32_sep
  left: ev_q32_lever_sep
  right: ev_q32_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_q32_ia
  left: ev_q32_lever_ia
  right: ev_q32_none_ia
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: unmatched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_fal_sep
  left: ev_fal_lever_sep
  right: ev_fal_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_yi_sep
  left: ev_yi_lever_sep
  right: ev_yi_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_olmo13_sep
  left: ev_olmo13_lever_sep
  right: ev_olmo13_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_olmo32_sep
  left: ev_olmo32_lever_sep
  right: ev_olmo32_none_sep
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 16
claims:
- id: k1
  text: a steering lever lowers obedience to instructions in tool output while the content stays usable; in Qwen3 it is strong but the sealed overall verdict is none
  location: abstract
  anchor: A steering lever, found by search
  jobs:
  - R1
  - R2
  supported_by:
  - c_q8_sep
  - c_q17_sep
  - c_smol_sep
  - c_gran_sep
  - c_olmo7_sep
  - c_phi4_sep
  - c_q14_sep
  - c_q32_sep
  - c_fal_sep
  - c_yi_sep
  - c_olmo13_sep
  - c_olmo32_sep
  - c_q8_ia
  - c_q17_ia
  - c_smol_ia
  - c_gran_ia
  - c_olmo7_ia
  - c_phi4_ia
  - c_q14_ia
  - c_q32_ia
  page: 1
  contradicted_by:
  - r_olmo7_lever_ia
  - r_gran_lever_ia
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: proxy
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Sections 3.7 and 3.8 tables
    R2attack: SEP utility probe under attack
    R2clean: not reported
schema_gaps:
- where: comparisons c_*_ia
  note: InjecAgent attack success is computed on each condition's valid replies, whose number changes with the lever (for example 81 to 93 of 300 for Qwen3-1.7B), so c3 is unmatched
- where: report
  note: the lever was chosen by search on Qwen3-8B, Qwen3-1.7B and two other models; the paper's sealed overall verdict is NONE (two of six models), and held-out families mostly fail the random-direction control; the study
    is described by its author as AI-run, and its number rechecks name a commercial assistant

`smalllocal_zenodo2026` (doi_10_5281_zenodo_23811220)

In [ ]:
%%writefile reports/records/smalllocal_zenodo2026.yaml
report:
  id: smalllocal_zenodo2026
  slug: doi_10_5281_zenodo_23811220
  title: 'Prompt-Injection Defences on a Small Local Agent: What Works, What It Costs, and How Optimistic Development Numbers Are'
  first_author: Syed Muhammad Hassan Raza
  venue: Zenodo preprint
  peer_reviewed: 'no'
  doi: 10.5281/zenodo.23811220
  version_read: Zenodo version
  first_date: '2026-10-30'
  read: F
  role: mixed
  code_release: github.com/smhassanraza1/hijack-proof-agent
  extraction_scope: 'Sections 4 and 6: Table 4 rows none, spotlighting and repeat the user''s prompt on the registered test cases, and the round-0 system-prompt rule on train cases (text); (plan-then-gate, the paper''s own
    defence, is a planner with a separate call gate and is system-level; the tool filter and the prompt-injection classifier are left out as a filter and a detector)'
defences:
- id: spot
  name: spotlighting (AgentDojo implementation)
  origin: other_report
  base_checkpoint: Qwen3.5-4B (local)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 15
  layout: AgentDojo agent; injection in tool outputs
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: supported
    agent_tool_result: supported
  anchor: Every defence on every test case, important_instructions
- id: rep
  name: repeat the user's prompt (AgentDojo implementation)
  origin: other_report
  base_checkpoint: Qwen3.5-4B (local)
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 15
  layout: AgentDojo agent; injection in tool outputs
  layout_compat: *id001
  anchor: Every defence on every test case, important_instructions
- id: rule
  name: 'system-prompt rule: tool data is not from the user, do not act on instructions in it, tell the user'
  origin: this_report
  base_checkpoint: Qwen3.5-4B (local)
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 8
  layout: AgentDojo agent; injection in tool outputs
  layout_compat: *id001
  anchor: The cheapest ﬁx is a system-prompt rule
evaluations:
- id: ev_none
  defence_id: none:Qwen3.5-4B (local)
  benchmark: AgentDojo, registered test cases (158 attacked, 47 normal), important_instructions
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo program checks
  layout_same_as_defence: 'yes'
  measured_by: smalllocal_zenodo2026
  page: 15
  decoding: not reported
  sample_size: 158 attacked, 47 normal (cases complete in both arms)
  baseline_kind: off_the_shelf
- id: ev_none_ben
  defence_id: none:Qwen3.5-4B (local)
  benchmark: AgentDojo, registered test cases (158 attacked, 47 normal), normal tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo program checks
  layout_same_as_defence: 'yes'
  measured_by: smalllocal_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '47'
  baseline_kind: off_the_shelf
- id: ev_spot
  defence_id: spot
  benchmark: AgentDojo, registered test cases (158 attacked, 47 normal), important_instructions
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo program checks
  layout_same_as_defence: 'yes'
  measured_by: smalllocal_zenodo2026
  page: 15
  decoding: not reported
  sample_size: 158 attacked, 47 normal (cases complete in both arms)
- id: ev_spot_ben
  defence_id: spot
  benchmark: AgentDojo, registered test cases (158 attacked, 47 normal), normal tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo program checks
  layout_same_as_defence: 'yes'
  measured_by: smalllocal_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '47'
- id: ev_rep
  defence_id: rep
  benchmark: AgentDojo, registered test cases (158 attacked, 47 normal), important_instructions
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo program checks
  layout_same_as_defence: 'yes'
  measured_by: smalllocal_zenodo2026
  page: 15
  decoding: not reported
  sample_size: 158 attacked, 47 normal (cases complete in both arms)
- id: ev_rep_ben
  defence_id: rep
  benchmark: AgentDojo, registered test cases (158 attacked, 47 normal), normal tasks
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo program checks
  layout_same_as_defence: 'yes'
  measured_by: smalllocal_zenodo2026
  page: 15
  decoding: not reported
  sample_size: '47'
- id: ev_tr_none
  defence_id: none:Qwen3.5-4B (local)
  benchmark: AgentDojo train cases, important_instructions
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo program checks
  layout_same_as_defence: 'yes'
  measured_by: smalllocal_zenodo2026
  page: 8
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_tr_rule
  defence_id: rule
  benchmark: AgentDojo train cases, important_instructions
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo program checks
  layout_same_as_defence: 'yes'
  measured_by: smalllocal_zenodo2026
  page: 8
  decoding: not reported
results:
- id: r_none_asr
  evaluation_id: ev_none
  quantity: R1
  coverage: direct
  metric_as_printed: attack success
  value: '38'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 4
- id: r_none_ua
  evaluation_id: ev_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '65'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 4
- id: r_none_u
  evaluation_id: ev_none_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: utility
  value: '81'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 4
- id: r_spot_asr
  evaluation_id: ev_spot
  quantity: R1
  coverage: direct
  metric_as_printed: attack success
  value: '34'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 4
- id: r_spot_ua
  evaluation_id: ev_spot
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '62'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 4
- id: r_spot_u
  evaluation_id: ev_spot_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: utility
  value: '85'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 4
- id: r_rep_asr
  evaluation_id: ev_rep
  quantity: R1
  coverage: direct
  metric_as_printed: attack success
  value: '19'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 4
- id: r_rep_ua
  evaluation_id: ev_rep
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '60'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 4
- id: r_rep_u
  evaluation_id: ev_rep_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: utility
  value: '77'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 4
- id: r_tr_none
  evaluation_id: ev_tr_none
  quantity: R1
  coverage: direct
  metric_as_printed: attack success, train cases
  value: '37'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 8
  location: Section 4, round 0
- id: r_tr_rule
  evaluation_id: ev_tr_rule
  quantity: R1
  coverage: direct
  metric_as_printed: attack success, train cases
  value: '39'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 8
  location: Section 4, round 0
comparisons:
- id: c_spot
  left: ev_spot
  right: ev_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 15
- id: c_spot_ben
  left: ev_spot_ben
  right: ev_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 15
- id: c_rep
  left: ev_rep
  right: ev_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 15
- id: c_rep_ben
  left: ev_rep_ben
  right: ev_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 15
- id: c_rule
  left: ev_tr_rule
  right: ev_tr_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 8
claims:
- id: k1
  text: a safety rule in the system prompt had no measurable effect on this model
  location: abstract
  anchor: A safety rule in the system prompt had no measurable effect
  jobs:
  - R1
  supported_by:
  - c_rule
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 4; Section 4 round 0
    R2clean: Table 4 (utility)
    R2attack: Table 4
schema_gaps:
- where: report
  note: the paper's main defence, plan-then-gate, sends unplanned state-changing calls to a separate checker and is left out as system-level; two identical undefended runs flipped 15% of attack outcomes, so differences under
    about 10 points are within noise, as the author notes

`crossdt_sbseg2026` (doi_10_5753_sbseg_estendido_2026_33794)

In [ ]:
%%writefile reports/records/crossdt_sbseg2026.yaml
report:
  id: crossdt_sbseg2026
  slug: doi_10_5753_sbseg_estendido_2026_33794
  title: A Cross-Lingual Security Evaluation of Prompt Injection Defenses in Small Language Models
  first_author: Pedro Fuziwara Filho
  venue: SBSeg 2026 Extended Proceedings (Workshop de Cibersegurança em IA)
  peer_reviewed: 'yes'
  doi: 10.5753/sbseg_estendido.2026.33794
  version_read: published workshop paper
  first_date: '2026-09-01'
  read: F
  role: re_measures
  extraction_scope: 'Table 1: DefensiveTokens trained by the authors in Portuguese, with and without, on three small models and five test languages (attack success only); (Table 2 breaks the undefended rate down by attack
    type and is not coded)'
defences:
- id: dt_llama
  name: DefensiveTokens (five soft tokens, prompt-tuned by the authors on Portuguese MedPT with injected prompts)
  origin: other_report
  base_checkpoint: Llama 3.2 3B
  axis1:
  - delimiters_markers
  axis2:
  - soft_tokens_frozen
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: '[INST] instruction, [DATA] patient question with the injection appended, [RESP]; five soft tokens prepended'
  layout_compat: &id001
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: untested
  anchor: in prompt-tuning mode, defining five soft tokens
- id: dt_qwen
  name: DefensiveTokens (five soft tokens, prompt-tuned by the authors on Portuguese MedPT with injected prompts)
  origin: other_report
  base_checkpoint: Qwen 2.5 7B
  axis1:
  - delimiters_markers
  axis2:
  - soft_tokens_frozen
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: '[INST] instruction, [DATA] patient question with the injection appended, [RESP]; five soft tokens prepended'
  layout_compat: *id001
  anchor: in prompt-tuning mode, defining five soft tokens
- id: dt_phi
  name: DefensiveTokens (five soft tokens, prompt-tuned by the authors on Portuguese MedPT with injected prompts)
  origin: other_report
  base_checkpoint: Phi-4 Mini
  axis1:
  - delimiters_markers
  axis2:
  - soft_tokens_frozen
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: '[INST] instruction, [DATA] patient question with the injection appended, [RESP]; five soft tokens prepended'
  layout_compat: *id001
  anchor: in prompt-tuning mode, defining five soft tokens
evaluations:
- id: ev_pt_llama_none
  defence_id: none:Llama 3.2 3B
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in Portuguese (training language) (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
  baseline_kind: off_the_shelf
- id: ev_pt_llama_dt
  defence_id: dt_llama
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in Portuguese (training language) (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
- id: ev_pt_qwen_none
  defence_id: none:Qwen 2.5 7B
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in Portuguese (training language) (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
  baseline_kind: off_the_shelf
- id: ev_pt_qwen_dt
  defence_id: dt_qwen
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in Portuguese (training language) (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
- id: ev_pt_phi_none
  defence_id: none:Phi-4 Mini
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in Portuguese (training language) (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
  baseline_kind: off_the_shelf
- id: ev_pt_phi_dt
  defence_id: dt_phi
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in Portuguese (training language) (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
- id: ev_es_llama_none
  defence_id: none:Llama 3.2 3B
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in Spanish (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
  baseline_kind: off_the_shelf
- id: ev_es_llama_dt
  defence_id: dt_llama
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in Spanish (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
- id: ev_es_qwen_none
  defence_id: none:Qwen 2.5 7B
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in Spanish (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
  baseline_kind: off_the_shelf
- id: ev_es_qwen_dt
  defence_id: dt_qwen
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in Spanish (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
- id: ev_es_phi_none
  defence_id: none:Phi-4 Mini
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in Spanish (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
  baseline_kind: off_the_shelf
- id: ev_es_phi_dt
  defence_id: dt_phi
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in Spanish (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
- id: ev_fr_llama_none
  defence_id: none:Llama 3.2 3B
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in French (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
  baseline_kind: off_the_shelf
- id: ev_fr_llama_dt
  defence_id: dt_llama
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in French (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
- id: ev_fr_qwen_none
  defence_id: none:Qwen 2.5 7B
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in French (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
  baseline_kind: off_the_shelf
- id: ev_fr_qwen_dt
  defence_id: dt_qwen
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in French (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
- id: ev_fr_phi_none
  defence_id: none:Phi-4 Mini
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in French (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
  baseline_kind: off_the_shelf
- id: ev_fr_phi_dt
  defence_id: dt_phi
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in French (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
- id: ev_en_llama_none
  defence_id: none:Llama 3.2 3B
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in English (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
  baseline_kind: off_the_shelf
- id: ev_en_llama_dt
  defence_id: dt_llama
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in English (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
- id: ev_en_qwen_none
  defence_id: none:Qwen 2.5 7B
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in English (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
  baseline_kind: off_the_shelf
- id: ev_en_qwen_dt
  defence_id: dt_qwen
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in English (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
- id: ev_en_phi_none
  defence_id: none:Phi-4 Mini
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in English (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
  baseline_kind: off_the_shelf
- id: ev_en_phi_dt
  defence_id: dt_phi
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in English (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
- id: ev_de_llama_none
  defence_id: none:Llama 3.2 3B
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in German (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
  baseline_kind: off_the_shelf
- id: ev_de_llama_dt
  defence_id: dt_llama
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in German (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
- id: ev_de_qwen_none
  defence_id: none:Qwen 2.5 7B
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in German (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
  baseline_kind: off_the_shelf
- id: ev_de_qwen_dt
  defence_id: dt_qwen
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in German (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
- id: ev_de_phi_none
  defence_id: none:Phi-4 Mini
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in German (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
  baseline_kind: off_the_shelf
- id: ev_de_phi_dt
  defence_id: dt_phi
  benchmark: MedPT test set (4,375 questions), seven static injection types, injection in German (NLLB translation)
  attacker_type: static
  attacker_access: black_box
  judge: language-specific substring (hack, pirat, gehackt) in the answer
  layout_same_as_defence: 'yes'
  measured_by: crossdt_sbseg2026
  page: 5
  decoding: greedy, max 256 new tokens
  sample_size: '4375'
results:
- id: r_pt_llama_none
  evaluation_id: ev_pt_llama_none
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '18.38'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_pt_llama_dt
  evaluation_id: ev_pt_llama_dt
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '3.86'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_pt_qwen_none
  evaluation_id: ev_pt_qwen_none
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '99.63'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_pt_qwen_dt
  evaluation_id: ev_pt_qwen_dt
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '3.54'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_pt_phi_none
  evaluation_id: ev_pt_phi_none
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '53.55'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_pt_phi_dt
  evaluation_id: ev_pt_phi_dt
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '18.48'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_es_llama_none
  evaluation_id: ev_es_llama_none
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '4.47'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_es_llama_dt
  evaluation_id: ev_es_llama_dt
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '0.46'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_es_qwen_none
  evaluation_id: ev_es_qwen_none
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '44.30'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_es_qwen_dt
  evaluation_id: ev_es_qwen_dt
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '0.23'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_es_phi_none
  evaluation_id: ev_es_phi_none
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '61.10'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_es_phi_dt
  evaluation_id: ev_es_phi_dt
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '25.96'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_fr_llama_none
  evaluation_id: ev_fr_llama_none
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '5.68'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_fr_llama_dt
  evaluation_id: ev_fr_llama_dt
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '3.34'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_fr_qwen_none
  evaluation_id: ev_fr_qwen_none
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '48.18'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_fr_qwen_dt
  evaluation_id: ev_fr_qwen_dt
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '3.36'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_fr_phi_none
  evaluation_id: ev_fr_phi_none
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '47.25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_fr_phi_dt
  evaluation_id: ev_fr_phi_dt
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '16.31'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_en_llama_none
  evaluation_id: ev_en_llama_none
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '9.85'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_en_llama_dt
  evaluation_id: ev_en_llama_dt
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '1.23'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_en_qwen_none
  evaluation_id: ev_en_qwen_none
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '99.20'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_en_qwen_dt
  evaluation_id: ev_en_qwen_dt
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '1.65'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_en_phi_none
  evaluation_id: ev_en_phi_none
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '66.95'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_en_phi_dt
  evaluation_id: ev_en_phi_dt
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '15.42'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_de_llama_none
  evaluation_id: ev_de_llama_none
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '1.18'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_de_llama_dt
  evaluation_id: ev_de_llama_dt
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '0.09'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_de_qwen_none
  evaluation_id: ev_de_qwen_none
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '47.09'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_de_qwen_dt
  evaluation_id: ev_de_qwen_dt
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '0.41'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_de_phi_none
  evaluation_id: ev_de_phi_none
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '44.69'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
- id: r_de_phi_dt
  evaluation_id: ev_de_phi_dt
  quantity: R1
  coverage: direct
  metric_as_printed: attack success rate
  value: '18.41'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table 1
comparisons:
- id: c_pt_llama
  left: ev_pt_llama_dt
  right: ev_pt_llama_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_pt_qwen
  left: ev_pt_qwen_dt
  right: ev_pt_qwen_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_pt_phi
  left: ev_pt_phi_dt
  right: ev_pt_phi_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_es_llama
  left: ev_es_llama_dt
  right: ev_es_llama_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_es_qwen
  left: ev_es_qwen_dt
  right: ev_es_qwen_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_es_phi
  left: ev_es_phi_dt
  right: ev_es_phi_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_fr_llama
  left: ev_fr_llama_dt
  right: ev_fr_llama_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_fr_qwen
  left: ev_fr_qwen_dt
  right: ev_fr_qwen_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_fr_phi
  left: ev_fr_phi_dt
  right: ev_fr_phi_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_en_llama
  left: ev_en_llama_dt
  right: ev_en_llama_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_en_qwen
  left: ev_en_qwen_dt
  right: ev_en_qwen_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_en_phi
  left: ev_en_phi_dt
  right: ev_en_phi_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_de_llama
  left: ev_de_llama_dt
  right: ev_de_llama_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_de_qwen
  left: ev_de_qwen_dt
  right: ev_de_qwen_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_de_phi
  left: ev_de_phi_dt
  right: ev_de_phi_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 5
claims:
- id: k1
  text: DefensiveTokens trained only in Portuguese give a robust defence against prompt injection in other languages
  location: abstract
  anchor: DefensiveTokens provide a robust defense against prompt injection attacks in multilingual applications
  jobs:
  - R1
  supported_by:
  - c_pt_llama
  - c_pt_qwen
  - c_pt_phi
  - c_es_llama
  - c_es_qwen
  - c_es_phi
  - c_fr_llama
  - c_fr_qwen
  - c_fr_phi
  - c_en_llama
  - c_en_qwen
  - c_en_phi
  - c_de_llama
  - c_de_qwen
  - c_de_phi
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 1
    R2clean: not reported
    R2attack: not reported
schema_gaps:
- where: defences dt_*
  note: the authors' DefensiveTokens run on base models with plain-text [INST]/[DATA]/[RESP] markers, not the chat-role format of the original release, so axis 1 is coded as delimiters for this report
- where: report
  note: no utility or answer-quality measure is reported for the defended models, so the claim of a robust defence rests on attack success alone

## Stage 2. The PDFs read are the PDFs registered, and every report is an included study

In [ ]:
import csv, logging, datetime as dt
from collections import Counter
import schema, search, corpus, paths
for m in (schema, search, corpus, paths):
    importlib.reload(m)
logging.getLogger("pypdf").setLevel(logging.ERROR)
from pypdf import PdfReader

BATCH = "04h"
cfg_exclusions = 2
cfg_next = "batch 04i"
IDS = ['backfire_zenodo2026', 'socportab_zenodo2026', 'transfact_zenodo2026', 'smallagentsq_zenodo2026', 'poisonedtools_zenodo2026', 'msacore_zenodo2026', 'whosewords_zenodo2026', 'whosaidthis_zenodo2026', 'smalllocal_zenodo2026', 'crossdt_sbseg2026']
man = {m["slug"]: m for m in search.read_csv(paths.REPORTS / "defences.csv")}
problems, texts = [], {}
for rid in IDS:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    pdf = paths.DATA_DEFENCES / f"{slug}.pdf"
    m = man.get(slug, {})
    if not pdf.exists():
        problems.append(f"{rid}: {pdf.name} not in data/defences"); continue
    sha = corpus.sha256(pdf)
    rd = PdfReader(str(pdf))
    texts[slug] = ([p.extract_text() or "" for p in rd.pages], [p.extract_text(extraction_mode="layout") or "" for p in rd.pages])
    ok_sha = sha == m.get("sha256")
    ok_inc = m.get("decision") == "include"
    print(f"{rid:<28} {slug:<12} pages {len(rd.pages):>3}  sha256 {'matches' if ok_sha else 'DIFFERS'}  decision {m.get('decision')}")
    if not ok_sha:
        problems.append(f"{rid}: PDF differs from the one registered in reports/defences.csv")
    if not ok_inc:
        problems.append(f"{rid}: not an included study")

## Stage 2b. Full-text exclusions found while coding

Each study below was included at title and abstract; its full text shows it does not meet the inclusion rule. The anchor is checked in its PDF, then the decision is written to schema/fulltext_decisions.csv and schema/screening_decisions.csv and carried into reports/screening.csv, reports/defences.csv, reports/fulltext_check.csv and reports/prisma_counts.json, as notebook 02d did.

In [ ]:
import json
import overlap
importlib.reload(overlap)
EXCL = [{'key': 'doi:10.5281/zenodo.23953154', 'slug': 'doi_10_5281_zenodo_23953154', 'reason': 'system level', 'anchor': 'measure four server-side defenses with the same scan suite', 'note': 'red-teaming of a composed RAG agent: the only defended build bundles four server-side controls (secrets removed from the prompt, document sanitising, an output guardrail and a file allow-list); no prompt-level defence is measured against no defence'}, {'key': 'doi:10.70715/jitcai.2026.v3.i3.071', 'slug': 'doi_10_70715_jitcai_2026_v3_i3_071', 'reason': 'system level', 'anchor': 'The combination of these mechanisms reflects a layered defence strategy', 'note': 'the only defended arm combines an input filter on user prompts with prompt hardening; prompt hardening is never measured on its own, and the malicious prompts are mostly direct injections into the user prompt'}]
FT_FIELDS = ["key", "slug", "decision", "reason", "basis", "anchor", "note"]
SD_FIELDS = ["key", "decision", "reason", "stage", "access_status", "note", "title"]
CHECK_FIELDS = ["key", "slug", "decision", "reason", "basis", "page", "status", "anchor", "sha256", "note"]
ft = search.read_csv(paths.SCHEMA / "fulltext_decisions.csv")
sd = search.read_csv(paths.SCHEMA / "screening_decisions.csv")
fchk = search.read_csv(paths.REPORTS / "fulltext_check.csv")
mlist = search.read_csv(paths.REPORTS / "defences.csv")
for e in EXCL:
    m = next((x for x in mlist if x["slug"] == e["slug"]), None)
    pdf = paths.DATA_DEFENCES / f"{e['slug']}.pdf"
    if m is None or not pdf.exists():
        problems.append(f"exclusion {e['slug']}: not in reports/defences.csv or PDF missing"); continue
    sha = corpus.sha256(pdf)
    if sha != m.get("sha256"):
        problems.append(f"exclusion {e['slug']}: PDF differs from the one registered")
    if e["reason"] not in search.EXCLUSION_REASONS:
        problems.append(f"exclusion {e['slug']}: reason not in the fixed list")
    pages = overlap.pdf_page_texts(pdf)
    a = overlap.norm_text(e["anchor"])
    hits = [i + 1 for i, t in enumerate(pages) if a in t]
    status = "found" if hits else "anchor not found"
    if not hits:
        problems.append(f"exclusion {e['slug']}: anchor not found in the PDF")
    row = {"key": e["key"], "slug": e["slug"], "decision": "exclude", "reason": e["reason"], "basis": "full text", "anchor": e["anchor"], "note": e["note"]}
    ft = [r for r in ft if r["key"] != e["key"]] + [row]
    fchk = [r for r in fchk if r["key"] != e["key"]] + [dict(row, page=str(hits[0]) if hits else "", status=status, sha256=sha)]
    n = 0
    for r in sd:
        if r["key"] == e["key"]:
            r.update(decision="exclude", reason=e["reason"], stage="full_text", note="full text: " + e["note"]); n += 1
    if n != 1:
        problems.append(f"exclusion {e['slug']}: {n} rows for {e['key']} in schema/screening_decisions.csv")
    m.update(decision="exclude", reason=e["reason"], stage="full_text", note="full text: " + e["note"])
    print(f"{e['slug']:<12} excluded at full text ({e['reason']}); anchor {status}{' on page ' + str(hits[0]) if hits else ''}")
def write_lf(path, rows, fields):              # the schema files were written with plain newlines; keep them that way
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore", lineterminator="\n")
        w.writeheader()
        w.writerows({k: r.get(k) or "" for k in fields} for r in rows)
write_lf(paths.SCHEMA / "fulltext_decisions.csv", ft, FT_FIELDS)
write_lf(paths.SCHEMA / "screening_decisions.csv", sd, SD_FIELDS)
search.write_csv(paths.REPORTS / "fulltext_check.csv", fchk, CHECK_FIELDS)
search.write_csv(paths.REPORTS / "defences.csv", mlist, list(mlist[0].keys()))
rows = search.read_csv(paths.REPORTS / "screening.csv")
rows, more = search.apply_decisions(rows, sd)
problems += more
search.write_csv(paths.REPORTS / "screening.csv", rows, search.SCREEN_FIELDS)
prisma = json.loads((paths.REPORTS / "prisma_counts.json").read_text())
prisma.update(search.prisma_counts(search.read_csv(paths.REPORTS / "search_log.csv"), rows))
prisma["fulltext_anchors_found"] = sum(1 for r in fchk if r["status"].startswith("found"))
(paths.REPORTS / "prisma_counts.json").write_text(json.dumps(prisma, indent=1))
print(f"included now: {prisma['included']}; full text assessed: {prisma['full_text_assessed']}; "
      f"full-text exclusions: {json.dumps(prisma['excluded_full_text_by_reason'])}")

## Stage 3. Validate against the frozen schema and the PDFs; grade the comparisons; extra checks

In [ ]:
import re
import pypdfium2 as pdfium
_docs = {}
def pdfium_page(slug, page):
    if slug not in _docs:
        _docs[slug] = pdfium.PdfDocument(str(paths.DATA_DEFENCES / f"{slug}.pdf"))
    return _docs[slug][page - 1].get_textpage().get_text_range()
S1 = schema.load_schema(paths.SCHEMA / "record_schema.yaml")
check_rows, summary, residual, third_rows = [], [], [], []
for rid in IDS:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    plain, lay = texts.get(slug, (None, None))
    probs, notes = schema.validate_record(rec, S1, plain, lay)
    kept, third = [], []                       # a value pypdf glues to its neighbour is looked for in a third extraction (pypdfium2)
    for p in probs:
        m = re.match(r"results:(\S+): value '(.+)' not found on page (\d+)$", p)
        if m and schema.value_on_page(m.group(2), pdfium_page(slug, int(m.group(3)))):
            third.append(f"{m.group(1)} ({m.group(2)!r}, page {m.group(3)})")
        else:
            kept.append(p)
    probs = kept
    for kind in ("defences", "evaluations", "results", "comparisons", "claims"):          # ids unique within each kind
        dup = [i for i, n in Counter(r["id"] for r in rec.get(kind) or []).items() if n > 1]
        probs += [f"{kind}: id {i} used more than once" for i in dup]
    fig_kept, third_chart = 0, []
    for r in rec.get("results") or []:                                                     # chart values kept as printed
        if r.get("value_source") == "figure_only" and r.get("value") not in (None, ""):
            fig_kept += 1
            pg = int(r["page"]) - 1
            if plain and not (schema.value_on_page(r["value"], plain[pg]) or schema.value_on_page(r["value"], lay[pg])):
                if schema.value_on_page(r["value"], pdfium_page(slug, int(r["page"]))):
                    third_chart.append(f"{r['id']} ({r['value']!r}, page {r['page']}, chart label)")
                else:
                    probs.append(f"results:{r['id']}: chart value {r['value']!r} not found on page {r['page']}")
    third_rows += [{"report": rid, "status": "value found in the third extraction", "detail": t} for t in third + third_chart]
    grades = Counter(schema.grade(c) for c in rec.get("comparisons") or [])
    n_val = sum(1 for r in rec.get("results") or [] if r.get("value_source") != "figure_only")
    n_anc = sum(1 for k in ("defences", "claims") for r in rec.get(k) or [] if r.get("anchor"))
    summary.append({"batch": BATCH, "report": rid, "slug": slug, "role": rec["report"]["role"],
                    "defences": len(rec.get("defences") or []), "evaluations": len(rec.get("evaluations") or []),
                    "results": len(rec.get("results") or []), "values_checked": n_val, "chart_values_checked": fig_kept,
                    "anchors_checked": n_anc, "comparisons": len(rec.get("comparisons") or []),
                    "grade_A": grades["A"], "grade_B": grades["B"], "grade_C": grades["C"], "grade_U": grades["U"],
                    "claims": len(rec.get("claims") or []),
                    **{f"cov_{q}": rec["coverage"][q] for q in ("R1", "R2clean", "R2attack", "R3ns", "R3cp", "R3safe")},
                    "values_third_extraction": len(third), "chart_values_third_extraction": len(third_chart), "residual_notes": len(notes), "problems": len(probs)})
    check_rows += [{"report": rid, "status": "problem", "detail": p} for p in probs]
    residual += [{"report": rid, "where": n.split(": ", 1)[0], "note": n.split(": ", 1)[1]} for n in notes]
    print(f"{rid:<28} problems {len(probs)}; values {n_val} ({len(third)} via the third extraction); chart values {fig_kept} ({len(third_chart)} via the third extraction); anchors {n_anc}; grades {dict(grades)}")
    for p in probs:
        print("   ", p)
    problems += [f"{rid}: {p}" for p in probs]
check_rows += third_rows + [{"report": g["report"], "status": "residual note", "detail": f"{g['where']}: {g['note']}"} for g in residual]
search.write_csv(paths.REPORTS / f"extraction_{BATCH}_check.csv", check_rows or [{"report": "all", "status": "ok", "detail": "no problems"}],
                 ["report", "status", "detail"])
search.write_csv(paths.REPORTS / f"extraction_{BATCH}_summary.csv", summary, list(summary[0].keys()))
for g in residual:
    print(f"  residual note, {g['report']}: {g['note'][:150]}")

## Stage 4. Batch register (one row per coded report, kept across batches)

In [ ]:
reg_path = paths.REPORTS / "extraction_batches.csv"
reg = [r for r in search.read_csv(reg_path) if r["batch"] != BATCH] if reg_path.exists() else []
today = dt.date.today().isoformat()
for s in summary:
    reg.append({"batch": BATCH, "report": s["report"], "slug": s["slug"], "coded": today,
                "status": "checked" if s["problems"] == 0 else "problems"})
reg.sort(key=lambda r: (r["batch"], r["report"]))
search.write_csv(reg_path, reg, ["batch", "report", "slug", "coded", "status"])
included = sum(1 for m in search.read_csv(paths.REPORTS / "defences.csv") if m.get("decision") == "include")   # read again: Stage 2b may change it
pilot = len(list(paths.RECORDS.glob("*.yaml"))) - len({r["report"] for r in reg})
print(f"records in reports/records: {len(list(paths.RECORDS.glob('*.yaml')))} (pilot {pilot}, batches {len({r['report'] for r in reg})}); included studies: {included}")

## Stage 5. Decision record and tests

In [ ]:
tot = {k: sum(s[k] for s in summary) for k in ("defences", "evaluations", "results", "values_checked", "values_third_extraction", "chart_values_checked",
                                              "chart_values_third_extraction", "anchors_checked", "comparisons", "grade_A", "grade_B", "grade_C", "grade_U", "claims")}
status = "BATCH CHECKED" if not problems else "BATCH NOT CHECKED: PROBLEMS BELOW"
lines = [f"decision_{BATCH}: {status}", f"this step: {dt.date.today().isoformat()}",
         "reports: " + ", ".join(IDS),
         f"records: {tot['defences']} defences, {tot['evaluations']} evaluations, {tot['results']} results, "
         f"{tot['comparisons']} comparisons, {tot['claims']} claims",
         f"checked against the PDFs: {tot['values_checked']} printed values ({tot['values_third_extraction']} of them found only in the third "
         f"extraction), {tot['chart_values_checked']} chart values ({tot['chart_values_third_extraction']} of them found only in the third extraction) "
         f"and {tot['anchors_checked']} anchors, each found on its page",
         f"comparison grades: A {tot['grade_A']}, B {tot['grade_B']}, C {tot['grade_C']}, U {tot['grade_U']}",
         f"residual notes: {len(residual)}; full-text exclusions found while coding: {cfg_exclusions}",
         f"next: {cfg_next}"] + [f"- problem: {p}" for p in problems]
(paths.REPORTS / f"decision_{BATCH}.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout.strip().splitlines()[-1] if tests.stdout.strip() else tests.stderr.strip()[-400:])

## Stage 6. Commit

In [ ]:
import os, subprocess
os.chdir(os.environ.get("SOURCEMAP_REPO_ROOT", "/content/drive/MyDrive/SourceMap_Research/sourcemap-research"))
r = subprocess.run(["python", "tools/commit_cell.py",
                    "04h extraction batch h: ten included studies coded from their PDFs and two full-text exclusions, values and anchors checked, comparisons graded"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)